# Session M1 — mesure décisive BLOC 10i (v7, DSO1 à 650–1000 m)

Notebook **autonome**, généré par `scripts/make_m1_colab_nb.py` (ne pas l'éditer à la main :
relancer le script). Protocole : `docs/M1_PROTOCOLE.md` (règle pré-enregistrée, avenants 1 et 2,
§ 5 et § 9, avenant du 2026-10-16).

**Lancer** : *Exécution → Modifier le type d'exécution → GPU* (A100 demandé par le § 9 pour que
le DSO4 reste comparable aux 91–104 ms de la v7 ; sur L4 la mesure est valable mais le DSO4 doit
être signalé comme non comparable), puis *Exécution → Tout exécuter*, accepter la fenêtre
d'autorisation Google Drive (compte qui contient `aneo_fno3d_runs`), et garder l'onglet ouvert
jusqu'au message `M1-e OK`. **Aucun secret** n'est utilisé. Ne pas lancer le notebook atom sur le
même environnement d'exécution.

| Cellule | Source |
|---|---|
| BLOC 0 | fno3d_v7.ipynb, BLOC 0 (cellule 78e2d31e), copie telle quelle |
| M1-a — Contrôle des fichiers figés, impression de la règle | ajoutee (§ 5.1 point 1) |
| BLOC 1 | fno3d_v7.ipynb, BLOC 1 (cellule a607a243), copie telle quelle |
| BLOC 2 | fno3d_v7.ipynb, BLOC 2 (cellule 4ed240f9), copie telle quelle |
| BLOC 3 | fno3d_v7.ipynb, BLOC 3 (cellule e392dd07), copie telle quelle |
| BLOC 4 | fno3d_v7.ipynb, BLOC 4 (cellule d969e4d3), copie telle quelle |
| BLOC 5 | fno3d_v7.ipynb, BLOC 5 (cellule 1e97ccea), copie telle quelle |
| BLOC 6 | fno3d_v7.ipynb, BLOC 6 (cellule d9650ae1), copie telle quelle |
| BLOC 7 | fno3d_v7.ipynb, BLOC 7 (cellule 9b8d220c), copie telle quelle |
| BLOC 8 | fno3d_v7.ipynb, BLOC 8 (cellule 8dfebd24), copie telle quelle |
| M1-b — À la place du BLOC 9 (aucun entraînement) | ajoutee : 2 lignes du BLOC 9, a l'identique |
| M1-c — BLOC 10 sans écrire dans eval_best_dso_v7/ | ajoutee |
| BLOC 10 | fno3d_v7.ipynb, BLOC 10 (cellule 966ddc8e), copie telle quelle |
| M1-d — Sorties de 10i dans eval_m1_bloc10i/ | ajoutee (§ 5.1 point 8) |
| BLOC 10i | cellule_10i.py, fichier entier, copie tel quel (sha256 du code a1fcc878ed3c6c6047538e5b48df25c6633c972a413d6bd42f415926599fbd6f) |
| M1-e — Archivage et contrôles de fin | ajoutee (§ 5.1 point 8) |
| M1-f — Jeu vierge graine 2027 (avenant n° 2, hors règle) | ajoutee (avenant n 2), gardee par la presence du jeu sur le Drive |

- Code des cellules copiées : **identique** à `fno3d_v7.ipynb` (sha256 du fichier source
  `62ec2c329f366fc6025e71e87c8bbe3264b73bfa220828f61e3ae528016d4de6`) et à `cellule_10i.py` (sha256 du code 10i `a1fcc878ed3c6c6047538e5b48df25c6633c972a413d6bd42f415926599fbd6f`). Sorties vidées.
- **Non exécutés** : BLOC 9 (risque de réentraînement, remplacé par M1-b), BLOCS 10c, 10d,
  10e, 10f, 10g, 10h (ajustements et variantes déjà vues sur le test : la correction est figée
  dans `couche_limite_v7.json`, § 4), BLOCS 11 et 12.
- **Écritures sur le Drive** : `aneo_fno3d_runs/v7/eval_m1_bloc10i/` seulement (le BLOC 3 recopie
  en outre `aneo_cache_v7/index.csv`, à l'identique). Rien n'est écrit dans `eval_best_dso_v7/`
  ni sur les JSON figés (vérifié par sha256 avant et après, M1-a et M1-e).
- La cellule 10i ne s'exécute **qu'une fois** (§ 5.2) : M1-a refuse de démarrer si
  `decision_10i.json` existe déjà.


## BLOC 0 — Montage du Drive, imports, GPU

In [ ]:
# BLOC 0 — Montage du Drive, imports, GPU
import os, json, math, time, random, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import gaussian_filter, binary_erosion
import pyarrow.csv as pacsv

TEST_MODE = os.environ.get("ANEO_TEST") == "1"      # validation locale sur mini-dataset

if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)
if device.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("BLOC 0 OK")

## M1-a — Contrôle des fichiers figés, impression de la règle

In [ ]:
# M1-a — Controle des fichiers figes et impression de la regle (AVANT TOUT CALCUL)
# Cellule ajoutee pour la session M1 (docs/M1_PROTOCOLE.md, § 5.1 point 1, avenant du 2026-10-16).
# Lit seulement le Drive, n'ecrit rien. S'arrete (exception) si un fichier fige manque ou a change,
# ou si une mesure 10i complete existe deja (regle d'execution unique, § 5.2).
import hashlib, datetime

M1_DRIVE_RUN = Path("/content/drive/MyDrive/aneo_fno3d_runs/v7")
M1_EVAL_DIR = M1_DRIVE_RUN / "eval_m1_bloc10i"
M1_SHA_PUBLIE = {                      # docs/M1_PROTOCOLE.md, § 4 et § 8
    "best_dso_v7.pt": "b2529442ee6fcdb18d3f35fe176663022604395743c161a25541ccd4e365e7de",
    "norm_stats.json": "7bae6748f270fa5e3f08a69a7f8e919f01d641d29b125981168cc15a03ee25f7",
}
M1_SHA_INFO = {"eval_best_dso_v7/test_summary.json": "7944ba0aa0311a08cd1e1555c1a225239b3deabc5c9c2b55dcf4406b5da17425"}   # § 8 (controle, non bloquant)
M1_SHA_10I = "a1fcc878ed3c6c6047538e5b48df25c6633c972a413d6bd42f415926599fbd6f"             # code de la cellule 10i (en-tete de cellule_10i.py)
M1_JSON_CORRECTION = ("correction_moyenne_v7.json", "geostrophique_v2_v7.json", "couche_limite_v7.json")


def m1_sha256(p, bloc=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(bloc), b""):
            h.update(b)
    return h.hexdigest()


_manque = [n for n in (*M1_SHA_PUBLIE, *M1_JSON_CORRECTION) if not (M1_DRIVE_RUN / n).exists()]
if _manque:
    raise FileNotFoundError(f"Fichiers figes absents de {M1_DRIVE_RUN} : {_manque} (§ 9 : a deposer avant la session)")
for _d in (M1_EVAL_DIR, M1_DRIVE_RUN / "eval_best_dso_v7"):
    if (_d / "decision_10i.json").exists():
        raise RuntimeError(f"{_d / 'decision_10i.json'} existe deja : une mesure 10i complete a eu lieu. "
                           "Regle d'execution unique (§ 5.2) : rapporter ce resultat, ne pas remesurer.")
t0 = time.time()
M1_SHA = {n: m1_sha256(M1_DRIVE_RUN / n) for n in (*M1_SHA_PUBLIE, *M1_JSON_CORRECTION, *M1_SHA_INFO)
          if (M1_DRIVE_RUN / n).exists()}
_ecart = [n for n, s in M1_SHA_PUBLIE.items() if M1_SHA[n] != s]
if _ecart:
    raise RuntimeError(f"sha256 different de la valeur publiee (§ 4) : {_ecart} -> ce ne sont pas les fichiers de la v7")
with open(M1_DRIVE_RUN / "norm_stats.json", encoding="utf-8") as f:
    assert "w_level_sd" in json.load(f), "norm_stats.json sans w_level_sd : le BLOC 5 le reecrirait"
with open(M1_DRIVE_RUN / "couche_limite_v7.json", encoding="utf-8") as f:
    assert abs(json.load(f)["ridge_alpha"] - 0.01) < 1e-12, "couche_limite_v7.json : alpha different de 0.01"
M1_GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun"
M1_DEBUT = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

M1_REGLE = """## Règle de décision pré-enregistrée (validée par Rayen le 2026-10-14, salon #325)

**Objet.** Décider si la correction post-modèle de la v7 (estimateur géostrophique v2 + ridge
par niveau prédisant le décalage moyen du niveau) est retenue. La règle est écrite et versionnée
**avant toute mesure** ; elle ne sera pas modifiée après coup.

- **C1 — test synoptique** (protocole DSO1, 120 instants, 50 % des capteurs) : pire niveau
  évalué u ET v après correction ≤ 0,5 m/s. IC 95 % bootstrap sur les instants (1000 tirages)
  RAPPORTÉ, borne haute publiée mais non utilisée comme seuil (le test a servi à la conception
  de la correction).
- **C2 — épisodes** (46 instants jamais utilisés pour le choix), Δ = RMSE après − RMSE avant,
  bootstrap APPARIÉ sur les instants (1000 tirages), pour u et v. Sur chaque niveau 650, 800
  et 1000 m : borne haute de l'IC 95 % de Δ ≤ 0 (amélioration significative). Sur chaque
  niveau évalué de 100 à 500 m : borne haute de l'IC 95 % de Δ ≤ +0,02 m/s (pas de
  dégradation notable).
- **C3** — DSO2 (moyenne ≤ 1e-4), DSO3 (rugosité dans [0,9 ; 1,1]), DSO5 (RMSE ≤ 0,7 à 25 %
  des capteurs) et DSO4 (temps total modèle + correction < 1 s par instant, GPU) restent
  conformes.
- **Pour information, hors règle** : cas 100 % capteurs, valeurs absolues avant/après avec IC.
- **Décision** : correction RETENUE ⇔ C1 ET C2 ET C3. Sinon : gel de v7 seule, DSO1 rapporté
  honnêtement (critère OMM). Méthode figée : ridge par niveau 10h, alpha = 0,01 choisi sur la
  validation, paramètres dans couche_limite_v7.json (sha256 imprimé par la cellule 10i). Pas
  de v8 sans accord de Rayen.
- **Mesure** : cellule BLOC 10i du notebook Colab fno3d_v7.ipynb, la règle étant imprimée
  avant tout calcul.

## Avenant n° 1 (2026-10-14, décision de Rayen, salon #332) — avant toute mesure

Objet : le mode de calcul de l'intervalle de confiance de **C2** (épisodes). Rien d'autre ne change :
seuils, niveaux, C1, C3 et la règle de décision restent ceux de la section précédente.

- **Constat** : les 46 instants d'épisodes ne sont pas indépendants. Ils tombent sur les jours de test
  4, 97 et 11, soit environ **3 situations** : 18 instants « rafales » à 1 min d'écart,
  18 « lever du soleil » et 10 « rafales 10 s ». Un bootstrap sur les instants sous-estime l'IC.
- **C2 utilise un bootstrap PAR BLOCS** : blocs de **6 instants consécutifs**, rééchantillonnés
  **à l'intérieur de chaque épisode**, en conservant la taille de chaque épisode (18 / 18 / 10).
  Le bootstrap reste apparié (même tirage avant / après), avec 1000 tirages. Pour C1, l'IC
  rapporté rééchantillonne le test synoptique par jours entiers.
- **Pour information, hors règle** : l'IC par instants est calculé et publié à côté.
- **Réserve, à reproduire telle quelle dans le rapport** : ce bootstrap stratifié est
  **conditionnel aux 3 situations**. Il corrige l'autocorrélation à l'intérieur d'un épisode,
  mais ne mesure pas la variabilité d'un épisode à l'autre. Avec 3 épisodes, aucune méthode ne
  le peut, et C2 a donc une **puissance limitée**.
- **Mise en œuvre** : cellule BLOC 10i, avec `BOOTSTRAP = "blocs"` et `BLOC_LEN = 6`, imprimés
  avec la règle avant tout calcul."""

print("=" * 96)
print(f"SESSION M1 — mesure decisive BLOC 10i | {M1_DEBUT} | GPU : {M1_GPU}")
print("Regle pre-enregistree : commit 0b64978 (salon #325) ; avenant n 1 : commit 33da965 (salon #332)")
print("=" * 96)
print(M1_REGLE)
print("=" * 96)
print(f"sha256 (calcules sur le Drive en {time.time() - t0:.0f} s) :")
for n, s in M1_SHA.items():
    etat = ""
    if n in M1_SHA_PUBLIE:
        etat = "  (= valeur publiee)"
    elif n in M1_SHA_INFO:
        etat = "  (= valeur publiee)" if s == M1_SHA_INFO[n] else "  (DIFFERENT de la valeur publiee § 8 : a signaler)"
    print(f"  {s}  {n}{etat}")
print(f"  {M1_SHA_10I}  code de la cellule 10i (publie ; recontrole apres execution par M1-e)")
if "A100" not in M1_GPU:
    print(f"ATTENTION : GPU {M1_GPU}, pas A100 : le DSO4 n'est pas comparable aux 91-104 ms de la v7 "
          "(§ 5.1 point 5, § 9). A signaler avec le verdict.")
print("M1-a OK")


## BLOC 1 — Configuration

In [ ]:
# BLOC 1 — Configuration
# Securite : si le BLOC 0 n'a pas ete execute (redemarrage de session), on recharge l'essentiel.
if "TEST_MODE" not in globals():
    import os, json, math, time, random, gc
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    TEST_MODE = os.environ.get("ANEO_TEST") == "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("BLOC 0 non execute : relance-le pour monter le Drive et charger tous les imports.")

if TEST_MODE:
    DATA_DIR = Path(os.environ["ANEO_DATA"])
    CACHE_DIR = Path(os.environ["ANEO_CACHE"])
    RUN_DIR = Path(os.environ["ANEO_RUN"])
else:
    DATA_DIR = Path("/content/drive/MyDrive/aneo_dataset_3d_v2")   # dataset genere v2 (csv.gz)
    CACHE_DIR = Path("/content/aneo_cache_v7")                      # tenseurs .npy (disque local)
    RUN_DIR = Path("/content/drive/MyDrive/aneo_fno3d_runs/v7")     # checkpoints, resultats, figures
for d in (CACHE_DIR, RUN_DIR, RUN_DIR / "figures"):
    d.mkdir(parents=True, exist_ok=True)

# Cache persistant sur le Drive (v7 : surface moyennee par blocs 2x2, incompatible avec le cache v6)
import shutil
CACHE_DRIVE = None if TEST_MODE else Path("/content/drive/MyDrive/aneo_cache_v7")

if CACHE_DRIVE is not None and (CACHE_DRIVE / "index.csv").exists() and not (CACHE_DIR / "index.csv").exists():
    print("Copie du cache depuis le Drive vers le disque local...")
    t0 = time.time()
    shutil.copytree(CACHE_DRIVE, CACHE_DIR, dirs_exist_ok=True)
    print(f"Cache copie en {(time.time() - t0) / 60:.1f} min")

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)
with open(DATA_DIR / "split_days.json", encoding="utf-8") as f:
    SPLIT = json.load(f)

# Geometrie (lue depuis meta.json du generateur)
NX_S, NY_S = META["grille_surface"]["nx"], META["grille_surface"]["ny"]
DX_S = float(META["grille_surface"]["dx_m"])
DY_S = float(META["grille_surface"]["dy_m"])
STRIDE = META["noeuds_3d"]["stride"]
NX, NY = NX_S // STRIDE, NY_S // STRIDE                     # grille du modele (100 m)
DX, DY = DX_S * STRIDE, DY_S * STRIDE
Z_LEVELS = np.array(META["noeuds_3d"]["z_levels_asl_m"], dtype=np.float64)
NZ = len(Z_LEVELS)
NODE_ID_START = META["noeuds_3d"]["point_id_debut"]
TERRAIN_SMOOTH_M = 300.0     # meme lissage que le generateur pour la condition au sol

# Modele
WIDTH = 48
MODES = (6, 32, 32)          # (z, y, x) : plus de modes pour les structures fines (DSO3)
N_LAYERS = 4
LOCAL_CONV = True            # branche convolutive locale 3x3x3 (details fins, DSO3)
POOL_SURFACE = True          # surface : moyenne des capteurs par bloc 2x2 au lieu d'1 capteur sur 4
PROJ_HIDDEN = 128            # largeur de la tete de projection
PAD = (4, 8, 8)              # rembourrage (z, y, x) : limite la periodicite de la FFT

# Entrainement
EPOCHS = 140
STEPS_PER_EPOCH = 400        # instants tires au hasard par epoque
VAL_SAMPLES = 64             # sous-ensemble fixe de validation
BATCH_SIZE = 1
ACCUM_STEPS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_PHYS_EPOCHS = 10      # montee progressive des termes physiques
NUM_WORKERS = 4
EMA_DECAY = 0.999             # moyenne exponentielle des poids (modele evalue et sauvegarde)

# Masquage des capteurs (entree)
KEEP_TRAIN = (0.2, 1.0)      # fraction de capteurs visibles, tiree a chaque echantillon
KEEP_EVAL = 0.5              # fraction fixe en validation / test

# Poids de la loss
LAMBDA_MASS = 1.0            # applique a L_mass / MASS_REF
MASS_REF = 5e-7              # echelle du terme de masse dans la loss (DSO2 avec marge)
LAMBDA_BC = 0.1
LAMBDA_SMOOTH = 0.0          # donnees deja lisses : terme desactive par defaut
LAMBDA_GRAD = 0.15           # loss sur les gradients horizontaux (corrige le lissage, DSO3 et w)
W_WEIGHT = 1.0               # poids de w dans L_data
W_LEVEL_MAX = 25.0           # plafond du poids de w par niveau (w est tres faible en altitude)
DSO_R2_ALL = 0.95            # objectif : R2 global >= 0.95 pour u, v ET w a chaque niveau

# Seuils des DSO
DSO1_RMSE = 0.5              # m/s, u et v, a chaque niveau
DSO1_R2 = 0.95               # R2 global par niveau
DSO2_RMS = 1e-4              # kg.m-3.s-1, RMS du residu de continuite
DSO3_RANGE = (0.0, 1.1)      # DSO3 : pas de bruit artificiel (rugosite <= 1.1 x verite)
DSO4_TIME = 1.0              # s par instant
DSO5_RMSE = 0.7              # m/s avec 25 % des capteurs
MIN_LEVEL_COVER = 0.05       # niveaux couvrant moins de 5 % du domaine exclus des DSO par niveau
F_CORIOLIS = 8.57e-5         # s-1, latitude ~36 N

if TEST_MODE:
    WIDTH, MODES, N_LAYERS, PAD, PROJ_HIDDEN = 4, (4, 8, 8), 2, (2, 4, 4), 4
    EPOCHS, STEPS_PER_EPOCH, VAL_SAMPLES, NUM_WORKERS = 2, 3, 2, 0
    WARMUP_PHYS_EPOCHS = 1

print(f"Grille modele : NZ={NZ} x NY={NY} x NX={NX} (dx={DX:.0f} m)")
print(f"Niveaux ASL   : {Z_LEVELS.tolist()}")
print(f"Split jours   : train={len(SPLIT['train_days'])} val={len(SPLIT['val_days'])} "
      f"test={len(SPLIT['test_days'])}")
print("BLOC 1 OK")

## BLOC 2 — Grille statique, masques, operateurs

In [ ]:
# BLOC 2 — Grille statique, masques, operateurs
grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv",
                   usecols=["point_id", "ix", "iy", "z_terrain_m", "z_rel_m", "z0_m",
                            "slope_deg", "aspect_deg"])

surf = grid[grid["point_id"] < NODE_ID_START].sort_values(["iy", "ix"]).reset_index(drop=True)
assert len(surf) == NX_S * NY_S, "Grille de surface incomplete"
nodes = grid[grid["point_id"] >= NODE_ID_START].reset_index(drop=True)

def s2d(col):
    return surf[col].values.reshape(NY_S, NX_S).astype(np.float64)

ZT_50 = s2d("z_terrain_m")
zt = ZT_50[::STRIDE, ::STRIDE]
z0 = np.maximum(s2d("z0_m")[::STRIDE, ::STRIDE], 1e-4)
slope = s2d("slope_deg")[::STRIDE, ::STRIDE]
aspect = np.radians(s2d("aspect_deg")[::STRIDE, ::STRIDE])
zrel_sensor = s2d("z_rel_m")[::STRIDE, ::STRIDE]

# point_id de surface -> indice plat (grille 50 m)
PID_TO_SURF = np.full(int(surf["point_id"].max()) + 1, -1, dtype=np.int64)
PID_TO_SURF[surf["point_id"].values] = np.arange(len(surf))

# noeud 3D -> indice plat (NZ, NY, NX)
z_abs_nodes = nodes["z_terrain_m"].values + nodes["z_rel_m"].values
k_nodes = np.abs(z_abs_nodes[:, None] - Z_LEVELS[None, :]).argmin(axis=1)
j_nodes = nodes["iy"].values // STRIDE
i_nodes = nodes["ix"].values // STRIDE
NODE_FLAT = np.full(int(nodes["point_id"].max()) - NODE_ID_START + 1, -1, dtype=np.int64)
NODE_FLAT[nodes["point_id"].values - NODE_ID_START] = (k_nodes * NY + j_nodes) * NX + i_nodes

VALID = np.zeros(NZ * NY * NX, dtype=bool)
VALID[NODE_FLAT[NODE_FLAT >= 0]] = True
VALID = VALID.reshape(NZ, NY, NX)

# Masque du residu de masse : les deux niveaux valides ET les voisins horizontaux utilises
# par les differences centrees valides (sinon le sol, rempli de zeros, fausse la divergence).
cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
VALID_INTERIOR = np.stack([binary_erosion(VALID[k], structure=cross, border_value=1)
                           for k in range(NZ)])
MASS_MASK = VALID_INTERIOR[1:] & VALID_INTERIOR[:-1]

# Premier noeud valide de chaque colonne (condition cinematique au sol)
FIRST_VALID = VALID.copy()
FIRST_VALID[1:] &= ~VALID[:-1]

# Pente du relief vu par l'ecoulement (meme lissage que le generateur)
zt_flow = gaussian_filter(ZT_50, sigma=TERRAIN_SMOOTH_M / DX_S, mode="nearest")[::STRIDE, ::STRIDE]
DZT_DY, DZT_DX = np.gradient(zt_flow, DY, DX)

# Champs statiques d'entree (normalises a la main, ordres de grandeur O(1))
z_asl = np.broadcast_to(Z_LEVELS[:, None, None], (NZ, NY, NX))
z_agl = np.clip(z_asl - zt[None], 0.0, None)
logz0 = np.log(z0)
STATIC_2D = np.stack([
    zt / 500.0,
    (logz0 - logz0.mean()) / (logz0.std() + 1e-6),
    slope / 10.0,
    np.sin(aspect),
    np.cos(aspect),
    (zrel_sensor - 25.0) / 5.0,
]).astype(np.float32)
STATIC_3D = np.stack([z_asl / 1000.0, z_agl / 1000.0, VALID.astype(np.float64)]).astype(np.float32)

N_SURF = 7                                   # u, v, w, T, rho, RH, p
# Coordonnees centrees et altitude des capteurs (estimation du vent geostrophique)
XC = (np.arange(NX) * DX)[None, :].repeat(NY, 0) - (NX - 1) * DX / 2
YC = (np.arange(NY) * DY)[:, None].repeat(NX, 1) - (NY - 1) * DY / 2
Z_SENSOR = (ZT_50 + s2d("z_rel_m")).reshape(NY, STRIDE, NX, STRIDE).mean(axis=(1, 3)) if POOL_SURFACE \
    else zt + zrel_sensor

def pool_surface(S):
    """(8, NY_S*NX_S ou NY_S, NX_S) -> (8, NY, NX). Canaux 0-6 : variables, canal 7 : capteur disponible.
    POOL_SURFACE : moyenne des capteurs disponibles de chaque bloc STRIDE x STRIDE (tous les capteurs
    sont utilises et le bruit de mesure est reduit) ; sinon un capteur sur STRIDE**2."""
    S = S.reshape(8, NY_S, NX_S)
    if not POOL_SURFACE:
        return S[:, ::STRIDE, ::STRIDE].copy()
    m = (S[7] > 0.5).astype(np.float32).reshape(NY, STRIDE, NX, STRIDE)
    cnt = m.sum(axis=(1, 3))
    out = np.zeros((8, NY, NX), dtype=np.float32)
    for c in range(7):
        v = S[c].reshape(NY, STRIDE, NX, STRIDE)
        out[c] = np.where(cnt > 0, (v * m).sum(axis=(1, 3)) / np.maximum(cnt, 1.0), 0.0)
    out[7] = (cnt > 0).astype(np.float32)
    return out

N_TFEAT = 8                    # heure (2), jour de l'annee (2), vent geostrophique (2), vent thermique (2)
C_IN = N_SURF + 1 + STATIC_2D.shape[0] + STATIC_3D.shape[0] + N_TFEAT
print(f"Noeuds valides : {VALID.sum():,} / {VALID.size:,} "
      f"| masse : {MASS_MASK.sum():,} demi-niveaux | canaux d'entree : {C_IN}")

# Operateurs discrets identiques au generateur (np.gradient + demi-niveaux)
DZ_T = torch.tensor(np.diff(Z_LEVELS), dtype=torch.float32, device=device)

def grad1d(f, h, dim):
    """Differences centrees a l'interieur, decentrees aux bords (equivalent np.gradient)."""
    n = f.shape[dim]
    first = (f.narrow(dim, 1, 1) - f.narrow(dim, 0, 1)) / h
    inner = (f.narrow(dim, 2, n - 2) - f.narrow(dim, 0, n - 2)) / (2.0 * h)
    last = (f.narrow(dim, n - 1, 1) - f.narrow(dim, n - 2, 1)) / h
    return torch.cat([first, inner, last], dim=dim)

def mass_residual(rho, u, v, w):
    """r(k+1/2) = (rho w[k+1] - rho w[k]) / dz + 0.5 (D[k] + D[k+1]) ; tenseurs (B, NZ, NY, NX)."""
    d = grad1d(rho * u, DX, -1) + grad1d(rho * v, DY, -2)
    rw = rho * w
    return (rw[:, 1:] - rw[:, :-1]) / DZ_T[None, :, None, None] + 0.5 * (d[:, 1:] + d[:, :-1])

VALID_T = torch.from_numpy(VALID).to(device)
GRAD_MASK_X_T = torch.from_numpy(VALID[:, :, 1:] & VALID[:, :, :-1]).to(device)
GRAD_MASK_Y_T = torch.from_numpy(VALID[:, 1:, :] & VALID[:, :-1, :]).to(device)
MASS_MASK_T = torch.from_numpy(MASS_MASK).to(device)
FIRST_VALID_T = torch.from_numpy(FIRST_VALID).to(device)
DZT_DX_T = torch.tensor(DZT_DX, dtype=torch.float32, device=device)
DZT_DY_T = torch.tensor(DZT_DY, dtype=torch.float32, device=device)
STATIC_2D_T = torch.from_numpy(STATIC_2D).to(device)
STATIC_3D_T = torch.from_numpy(STATIC_3D).to(device)
del grid
gc.collect()
print("BLOC 2 OK")

## BLOC 3 — Construction du cache de tenseurs (une seule fois)

In [ ]:
# BLOC 3 — Construction du cache de tenseurs (une seule fois)
# Pour chaque instant :
#   <jour>_<HHMMSS>_S.npy : surface (8, NY, NX) = u, v, w, T, rho, RH, p, capteur_disponible
#   <jour>_<HHMMSS>_Y.npy : volume (4, NZ, NY, NX) = u, v, w, rho (0 sous le terrain)
READ_COLS = ["point_id", "timestamp_utc", "temperature_c", "air_density_kg_m3",
             "relative_humidity_pct", "pressure_pa", "wind_u_m_s", "wind_v_m_s",
             "wind_w_m_s", "mask_i_obs"]
SURF_VARS = ["wind_u_m_s", "wind_v_m_s", "wind_w_m_s", "temperature_c",
             "air_density_kg_m3", "relative_humidity_pct", "pressure_pa"]

def day_of(path):
    return int(path.name.split("_")[3])

def split_of(day):
    if day in SPLIT["train_days"]:
        return "train"
    if day in SPLIT["val_days"]:
        return "val"
    return "test"

def build_day(path):
    day = day_of(path)
    opts = pacsv.ConvertOptions(include_columns=READ_COLS,
                                column_types={"timestamp_utc": "string"})
    tbl = pacsv.read_csv(path, convert_options=opts)
    pid = tbl.column("point_id").to_numpy()
    ts = tbl.column("timestamp_utc").to_pandas()
    codes, uniq = pd.factorize(ts, sort=True)
    cols = {c: tbl.column(c).to_numpy().astype(np.float32) for c in READ_COLS[2:]}
    del tbl
    rows = []
    for c, stamp in enumerate(uniq):
        sel = np.nonzero(codes == c)[0]
        p = pid[sel]
        is_s = p < NODE_ID_START
        ps, pn = p[is_s], p[~is_s]
        sel_s, sel_n = sel[is_s], sel[~is_s]

        S = np.zeros((8, NY_S * NX_S), dtype=np.float32)
        flat = PID_TO_SURF[ps]
        for ci, var in enumerate(SURF_VARS):
            S[ci, flat] = cols[var][sel_s]
        S[7, flat] = cols["mask_i_obs"][sel_s]
        S = pool_surface(S)

        Y = np.zeros((4, NZ * NY * NX), dtype=np.float32)
        nflat = NODE_FLAT[pn - NODE_ID_START]
        for ci, var in enumerate(["wind_u_m_s", "wind_v_m_s", "wind_w_m_s", "air_density_kg_m3"]):
            Y[ci, nflat] = cols[var][sel_n]
        Y = Y.reshape(4, NZ, NY, NX)

        tstamp = pd.Timestamp(stamp)
        stem = f"{day:03d}_{tstamp.strftime('%H%M%S')}"
        np.save(CACHE_DIR / f"{stem}_S.npy", S)
        np.save(CACHE_DIR / f"{stem}_Y.npy", Y)
        rows.append({"stem": stem, "day": day, "timestamp": str(tstamp), "split": split_of(day),
                     "hour": tstamp.hour + tstamp.minute / 60 + tstamp.second / 3600,
                     "doy": tstamp.dayofyear})
    return rows

index_path = CACHE_DIR / "index.csv"
done_days = set(pd.read_csv(index_path)["day"]) if index_path.exists() else set()
day_files = sorted(DATA_DIR.glob("aneo_observations_day_*_multitime.csv*"))
day_files = [p for p in day_files if not p.name.endswith(".part")]
todo = [p for p in day_files if day_of(p) not in done_days]
print(f"Jours disponibles : {len(day_files)} | deja en cache : {len(done_days)} | a traiter : {len(todo)}")

t0 = time.time()
for n, path in enumerate(todo, 1):
    rows = build_day(path)
    df_new = pd.DataFrame(rows)
    df_new.to_csv(index_path, mode="a", header=not index_path.exists(), index=False)
    el = time.time() - t0
    print(f"  jour {day_of(path):03d} : {len(rows)} instants | {n}/{len(todo)} "
          f"| reste ~{el / n * (len(todo) - n) / 60:.0f} min")
    gc.collect()

INDEX = pd.read_csv(index_path).drop_duplicates("stem").sort_values("stem").reset_index(drop=True)
print(INDEX["split"].value_counts().to_dict())

# Sauvegarde du cache sur le Drive (uniquement les fichiers absents)
if CACHE_DRIVE is not None:
    CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
    existing = {p_.name for p_ in CACHE_DRIVE.glob("*.npy")}
    new_files = [p_ for p_ in sorted(CACHE_DIR.glob("*.npy")) if p_.name not in existing]
    if new_files:
        print(f"Sauvegarde du cache sur le Drive : {len(new_files)} fichiers...")
        for p_ in new_files:
            shutil.copy2(p_, CACHE_DRIVE / p_.name)
    shutil.copy2(CACHE_DIR / "index.csv", CACHE_DRIVE / "index.csv")
print("BLOC 3 OK")

## BLOC 4 — Controle de la verite terrain

In [ ]:
# BLOC 4 — Controle de la verite terrain
# Le residu de masse et la condition au sol doivent etre ~0 sur les donnees : sinon la loss
# physique contredirait les donnees et DSO2 serait inatteignable.
def load_y(stem):
    return torch.from_numpy(np.load(CACHE_DIR / f"{stem}_Y.npy")).to(device)

checks = []
for stem in INDEX["stem"].sample(min(5, len(INDEX)), random_state=0):
    y = load_y(stem)[None]
    u, v, w, rho = y[:, 0], y[:, 1], y[:, 2], y[:, 3]
    r = mass_residual(rho, u, v, w)
    l_mass = (r[:, MASS_MASK_T] ** 2).mean().item()
    w_bc = u * DZT_DX_T + v * DZT_DY_T
    l_bc = ((w - w_bc)[:, FIRST_VALID_T] ** 2).mean().item()
    checks.append((stem, l_mass, l_bc))
    print(f"  {stem} : L_mass(verite) = {l_mass:.2e} | ecart condition sol = {l_bc:.2e} (m/s)^2")
assert max(c[1] for c in checks) < 1e-6, "La verite ne respecte pas la continuite : verifier le dataset"
print("BLOC 4 OK")

## BLOC 5 — Normalisation (jours d'entrainement uniquement)

In [ ]:
# BLOC 5 — Normalisation (jours d'entrainement uniquement)
stats_path = RUN_DIR / "norm_stats.json"
if stats_path.exists():
    with open(stats_path, encoding="utf-8") as f:
        NORM = json.load(f)
else:
    tr = INDEX[INDEX["split"] == "train"]["stem"]
    tr = tr.sample(min(200, len(tr)), random_state=0)
    s_sum = np.zeros(N_SURF); s_sq = np.zeros(N_SURF); s_n = 0
    y_sum = np.zeros(3); y_sq = np.zeros(3); y_n = 0
    for stem in tr:
        S = np.load(CACHE_DIR / f"{stem}_S.npy").astype(np.float64)
        m = S[7] > 0.5
        s_sum += S[:N_SURF, m].sum(axis=1); s_sq += (S[:N_SURF, m] ** 2).sum(axis=1); s_n += m.sum()
        Y = np.load(CACHE_DIR / f"{stem}_Y.npy").astype(np.float64)
        y_sum += Y[:3, VALID].sum(axis=1); y_sq += (Y[:3, VALID] ** 2).sum(axis=1); y_n += VALID.sum()
    s_mu = s_sum / s_n; s_sd = np.sqrt(np.maximum(s_sq / s_n - s_mu ** 2, 1e-12))
    y_mu = y_sum / y_n; y_sd = np.sqrt(np.maximum(y_sq / y_n - y_mu ** 2, 1e-12))
    NORM = {"surf_mu": s_mu.tolist(), "surf_sd": s_sd.tolist(),
            "y_mu": y_mu.tolist(), "y_sd": y_sd.tolist(), "surf_vars": SURF_VARS}
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(NORM, f, indent=2)

if "w_level_sd" not in NORM:
    tr = INDEX[INDEX["split"] == "train"]["stem"]
    tr = tr.sample(min(200, len(tr)), random_state=0)
    wl_sum = np.zeros(NZ); wl_sq = np.zeros(NZ); wl_n = np.zeros(NZ)
    for stem in tr:
        Yw = np.load(CACHE_DIR / f"{stem}_Y.npy")[2].astype(np.float64)
        for k in range(NZ):
            vals = Yw[k][VALID[k]]
            wl_sum[k] += vals.sum(); wl_sq[k] += (vals ** 2).sum(); wl_n[k] += vals.size
    wl_n = np.maximum(wl_n, 1)
    NORM["w_level_sd"] = np.sqrt(np.maximum(wl_sq / wl_n - (wl_sum / wl_n) ** 2, 1e-12)).tolist()
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(NORM, f, indent=2)

SURF_MU = torch.tensor(NORM["surf_mu"], dtype=torch.float32, device=device)[:, None, None]
SURF_SD = torch.tensor(NORM["surf_sd"], dtype=torch.float32, device=device)[:, None, None]
Y_MU = torch.tensor(NORM["y_mu"], dtype=torch.float32, device=device)
Y_SD = torch.tensor(NORM["y_sd"], dtype=torch.float32, device=device)
# poids de w par niveau : chaque niveau compte selon sa propre variabilite (sinon w en altitude,
# tres faible, est neglige par la loss et son R2 chute)
W_LEVEL_SD = torch.tensor(NORM["w_level_sd"], dtype=torch.float32, device=device)
W_LEVEL_W = torch.clamp((Y_SD[2] / W_LEVEL_SD) ** 2, 1.0, W_LEVEL_MAX)
print("Poids de w par niveau :", np.round(W_LEVEL_W.cpu().numpy(), 2))
print("Cible  mu :", np.round(NORM["y_mu"], 3), " sd :", np.round(NORM["y_sd"], 3))
print("BLOC 5 OK")

## BLOC 6 — Dataset, DataLoaders, assemblage de l'entree

In [ ]:
# BLOC 6 — Dataset, DataLoaders, assemblage de l'entree
def geostrophic_from_surface(S, sensor):
    """
    Vent geostrophique estime a partir des capteurs visibles :
    pression reduite au niveau de la mer, plan ajuste par moindres carres, equilibre geostrophique
    ug = -(1 / rho f) dp/dy ; vg = (1 / rho f) dp/dx.
    """
    m = sensor
    if m.sum() < 100:
        return 0.0, 0.0
    p = S[6][m].astype(np.float64)
    t_k = S[3][m].astype(np.float64) + 273.15
    z = Z_SENSOR[m]
    p_sl = p * np.exp(9.80665 * z / (287.05 * (t_k + 0.5 * 0.0065 * z)))
    A = np.stack([np.ones(m.sum()), XC[m], YC[m]], axis=1)
    coef, *_ = np.linalg.lstsq(A, p_sl, rcond=None)
    rho = float(S[4][m].mean())
    return -coef[2] / (rho * F_CORIOLIS), coef[1] / (rho * F_CORIOLIS)

def thermal_wind_from_surface(S, sensor):
    """
    Vent thermique (variation du vent geostrophique avec l'altitude, en s-1) estime a partir
    du gradient horizontal de temperature des capteurs visibles, ramenee au niveau de la mer :
    d(ug, vg)/dz = (g / (f T)) * (-dT/dy, dT/dx).
    """
    m = sensor
    if m.sum() < 100:
        return 0.0, 0.0
    t_red = S[3][m].astype(np.float64) + 0.0065 * Z_SENSOR[m]
    A = np.stack([np.ones(m.sum()), XC[m], YC[m]], axis=1)
    coef, *_ = np.linalg.lstsq(A, t_red, rcond=None)
    k = 9.80665 / (F_CORIOLIS * (float(t_red.mean()) + 273.15))
    return -k * coef[2], k * coef[1]

def time_geo_features(S, sensor, hour, doy):
    ug, vg = geostrophic_from_surface(S, sensor)
    su, sv = thermal_wind_from_surface(S, sensor)
    return np.array([math.sin(2 * math.pi * hour / 24), math.cos(2 * math.pi * hour / 24),
                     math.sin(2 * math.pi * doy / 365), math.cos(2 * math.pi * doy / 365),
                     ug / 10.0, vg / 10.0, su * 200.0, sv * 200.0], dtype=np.float32)

class ANEO3D(Dataset):
    """Renvoie la surface brute, les caracteristiques temporelles, la cible et le masque capteurs."""

    def __init__(self, stems, hours, doys, train, n_samples=None, seed=0):
        self.stems, self.hours, self.doys = list(stems), list(hours), list(doys)
        self.train = train
        self.n_samples = n_samples
        self.seed = seed

    def __len__(self):
        return self.n_samples if self.n_samples else len(self.stems)

    def __getitem__(self, i):
        if self.train:
            j = np.random.randint(len(self.stems))
            rng = np.random.default_rng()
            keep = rng.uniform(*KEEP_TRAIN)
        else:
            j = i
            rng = np.random.default_rng(self.seed + i)
            keep = KEEP_EVAL
        stem = self.stems[j]
        S = np.load(CACHE_DIR / f"{stem}_S.npy")
        Y = np.load(CACHE_DIR / f"{stem}_Y.npy")
        sensor = (S[7] > 0.5) & (rng.random(S[7].shape) < keep)
        tfeat = time_geo_features(S, sensor, self.hours[j], self.doys[j])
        return (torch.from_numpy(S[:N_SURF]), torch.from_numpy(sensor.astype(np.float32)),
                torch.from_numpy(tfeat), torch.from_numpy(Y), j)

def assemble_input(s_raw, sensor, tfeat):
    """Construit l'entree (B, C_IN, NZ, NY, NX) sur le GPU."""
    B = s_raw.shape[0]
    s = (s_raw - SURF_MU) / SURF_SD * sensor[:, None]
    s2d = torch.cat([s, sensor[:, None], STATIC_2D_T.expand(B, -1, -1, -1)], dim=1)
    x2d = s2d[:, :, None].expand(-1, -1, NZ, -1, -1)
    x3d = STATIC_3D_T.expand(B, -1, -1, -1, -1)
    xt = tfeat[:, :, None, None, None].expand(-1, -1, NZ, NY, NX)
    return torch.cat([x2d, x3d, xt], dim=1)

def subset(split):
    d = INDEX[INDEX["split"] == split]
    return d["stem"].values, d["hour"].values, d["doy"].values

tr_s, tr_h, tr_d = subset("train")
va_s, va_h, va_d = subset("val")
te_s, te_h, te_d = subset("test")
# Validation : instants synoptiques (toutes les 3 h) repartis regulierement sur les jours de validation
_ts_val = INDEX.set_index("stem").loc[va_s, "timestamp"]
_is_syn = np.array([(lambda t: t.minute == 0 and t.second == 0 and t.hour % 3 == 0)(pd.Timestamp(x))
                    for x in _ts_val])
_cand = np.nonzero(_is_syn)[0] if _is_syn.any() else np.arange(len(va_s))
vsel = _cand[np.unique(np.linspace(0, len(_cand) - 1, min(VAL_SAMPLES, len(_cand))).astype(int))]

train_ds = ANEO3D(tr_s, tr_h, tr_d, train=True, n_samples=STEPS_PER_EPOCH * BATCH_SIZE)
val_ds = ANEO3D(va_s[vsel], va_h[vsel], va_d[vsel], train=False, seed=1000)
test_ds = ANEO3D(te_s, te_h, te_d, train=False, seed=2000)

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                 persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, **loader_kw)
print(f"Instants : train={len(tr_s)} | val={len(vsel)} (sur {len(va_s)}) | test={len(te_s)}")
print("BLOC 6 OK")

## BLOC 7 — Modele PC-FNO3D

In [ ]:
# BLOC 7 — Modele PC-FNO3D
class SpectralConv3d(nn.Module):
    """Couche de Fourier 3D : poids complexes appris sur les modes conserves (z, y, x)."""

    def __init__(self, cin, cout, modes):
        super().__init__()
        self.cout = cout
        self.m1, self.m2, self.m3 = modes
        scale = 1.0 / (cin * cout)
        self.w = nn.ParameterList([
            nn.Parameter(scale * torch.randn(cin, cout, *modes, dtype=torch.cfloat))
            for _ in range(4)])

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, _, D, H, W = x.shape
            xf = torch.fft.rfftn(x, dim=(-3, -2, -1))
            out = torch.zeros(B, self.cout, D, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
            m1, m2, m3 = self.m1, self.m2, self.m3
            mul = lambda a, w: torch.einsum("bixyz,ioxyz->boxyz", a, w)
            out[:, :, :m1, :m2, :m3] = mul(xf[:, :, :m1, :m2, :m3], self.w[0])
            out[:, :, -m1:, :m2, :m3] = mul(xf[:, :, -m1:, :m2, :m3], self.w[1])
            out[:, :, :m1, -m2:, :m3] = mul(xf[:, :, :m1, -m2:, :m3], self.w[2])
            out[:, :, -m1:, -m2:, :m3] = mul(xf[:, :, -m1:, -m2:, :m3], self.w[3])
            return torch.fft.irfftn(out, s=(D, H, W), dim=(-3, -2, -1))


class PCFNO3D(nn.Module):
    def __init__(self, cin, width, modes, n_layers, pad, proj_hidden=128, local=False):
        super().__init__()
        self.pad = pad
        self.lift = nn.Sequential(nn.Conv3d(cin, width, 1), nn.GELU(), nn.Conv3d(width, width, 1))
        self.spec = nn.ModuleList([SpectralConv3d(width, width, modes) for _ in range(n_layers)])
        self.skip = nn.ModuleList([nn.Conv3d(width, width, 1) for _ in range(n_layers)])
        # Branche locale 3x3x3 : la couche de Fourier ne garde que les grandes echelles (modes tronques),
        # la convolution locale restitue les details fins (vents de pente, contrastes de rugosite).
        self.local = nn.ModuleList([nn.Conv3d(width, width, 3, padding=1) for _ in range(n_layers)]) \
            if local else None
        self.proj = nn.Sequential(nn.Conv3d(width, proj_hidden, 1), nn.GELU(),
                                  nn.Conv3d(proj_hidden, 3, 1))

    def forward(self, x):
        D, H, W = x.shape[-3:]
        pz, py, px = self.pad
        x = self.lift(x)
        x = F.pad(x, (0, px, 0, py, 0, pz))
        for n, (sc, sk) in enumerate(zip(self.spec, self.skip)):
            y = sc(x) + sk(x)
            if self.local is not None:
                y = y + self.local[n](x)
            x = y
            if n < len(self.spec) - 1:
                x = F.gelu(x)
        x = x[..., :D, :H, :W]
        # Tete de projection en fp32 : en bf16 la sortie est arrondie a ~3 chiffres (quelques cm/s),
        # ce bruit passe dans les differences centrees et cree un plancher sur le residu de masse (DSO2).
        with torch.autocast(device_type=x.device.type, enabled=False):
            return self.proj(x.float())   # (B, 3, NZ, NY, NX) : u, v, w normalises


model = PCFNO3D(C_IN, WIDTH, MODES, N_LAYERS, PAD, PROJ_HIDDEN, local=LOCAL_CONV).to(device)
n_par = sum(p.numel() * (2 if p.is_complex() else 1) for p in model.parameters())
print(f"Parametres : {n_par / 1e6:.2f} M")
print("BLOC 7 OK")

## BLOC 8 — Loss physique

In [ ]:
# BLOC 8 — Loss physique
def denorm(pred):
    """(B, 3, ...) normalise -> u, v, w physiques (m/s)."""
    return [pred[:, c] * Y_SD[c] + Y_MU[c] for c in range(3)]

def compute_loss(pred, y, phys_w):
    """
    pred : (B, 3, NZ, NY, NX) normalise ; y : (B, 4, NZ, NY, NX) physique (u, v, w, rho)
    L_data  : MSE normalisee sur les noeuds valides
    L_mass  : residu de continuite (unites physiques), meme operateur que le generateur
    L_bc    : w = u.grad(z_t) au premier noeud valide de chaque colonne
    L_smooth: gradients horizontaux de u, v (optionnel)
    """
    pred = pred.float()
    y_n = (y[:, :3] - Y_MU[None, :, None, None, None]) / Y_SD[None, :, None, None, None]
    err = (pred - y_n) ** 2
    err = torch.cat([err[:, :2], err[:, 2:3] * W_WEIGHT * W_LEVEL_W[None, None, :, None, None]], dim=1)
    l_data = err[:, :, VALID_T].mean()

    u, v, w = denorm(pred)
    rho = y[:, 3]
    r = mass_residual(rho, u, v, w)
    l_mass = (r[:, MASS_MASK_T] ** 2).mean()

    w_bc = u * DZT_DX_T + v * DZT_DY_T
    l_bc = (((w - w_bc) / Y_SD[2])[:, FIRST_VALID_T] ** 2).mean()

    # Loss sur les gradients horizontaux : erreur relative des variations entre cellules voisines
    l_grad = torch.zeros((), device=pred.device)
    if LAMBDA_GRAD > 0:
        gxp, gxt = pred[..., 1:] - pred[..., :-1], y_n[..., 1:] - y_n[..., :-1]
        gyp, gyt = pred[..., 1:, :] - pred[..., :-1, :], y_n[..., 1:, :] - y_n[..., :-1, :]
        num = ((gxp - gxt) ** 2)[:, :, GRAD_MASK_X_T].sum() + ((gyp - gyt) ** 2)[:, :, GRAD_MASK_Y_T].sum()
        den = (gxt ** 2)[:, :, GRAD_MASK_X_T].sum() + (gyt ** 2)[:, :, GRAD_MASK_Y_T].sum() + 1e-12
        l_grad = num / den

    l_smooth = torch.zeros((), device=pred.device)
    if LAMBDA_SMOOTH > 0:
        l_smooth = ((pred[:, :2, :, :, 1:] - pred[:, :2, :, :, :-1]) ** 2).mean() + \
                   ((pred[:, :2, :, 1:, :] - pred[:, :2, :, :-1, :]) ** 2).mean()

    total = l_data + LAMBDA_GRAD * l_grad + phys_w * (LAMBDA_MASS * l_mass / MASS_REF + LAMBDA_BC * l_bc
                                                      + LAMBDA_SMOOTH * l_smooth)
    return total, {"data": l_data.item(), "mass": l_mass.item(), "bc": l_bc.item(),
                   "smooth": l_smooth.item(), "grad": l_grad.item()}

def r2_per_component(pred, y):
    """R^2 par composante sur les noeuds valides, en unites physiques."""
    u, v, w = denorm(pred.float())
    out = []
    for c, p in enumerate((u, v, w)):
        t = y[:, c][:, VALID_T]
        q = p[:, VALID_T]
        ss_res = ((q - t) ** 2).sum()
        ss_tot = ((t - t.mean()) ** 2).sum()
        out.append((1 - ss_res / (ss_tot + 1e-12)).item())
    return out

def roughness(a):
    """Rugosite |dA/dx| + |dA/dy| moyennee sur les paires de voisins VALIDES ; a : (B, NZ, NY, NX)."""
    gx = (a[..., :, 1:] - a[..., :, :-1]).abs()[:, GRAD_MASK_X_T]
    gy = (a[..., 1:, :] - a[..., :-1, :]).abs()[:, GRAD_MASK_Y_T]
    return gx.mean() + gy.mean()

print("BLOC 8 OK")

## M1-b — À la place du BLOC 9 (aucun entraînement)

In [ ]:
# M1-b — A LA PLACE DU BLOC 9 (non execute) : aucun entrainement
# Le BLOC 9 reprend depuis last.pt (~7 Go, Drive). Si ce fichier manque, il REENTRAINE 140 epoques
# (~5 h) et reecrit best.pt et history.json sur le Drive : il n'est donc pas execute.
# Seules les deux lignes du BLOC 9 dont le BLOC 10 et la cellule 10i ont besoin sont reprises,
# a l'identique. Le checkpoint est charge par le BLOC 10 lui-meme, comme dans le notebook v7 :
# RUN_DIR/best_dso_v7.pt (sha256 verifie par M1-a et M1-c).
amp_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
use_amp = device.type == "cuda"
print(f"BLOC 9 non execute | amp_dtype = {amp_dtype} | use_amp = {use_amp}")
print("M1-b OK")


## M1-c — BLOC 10 sans écrire dans eval_best_dso_v7/

In [ ]:
# M1-c — Le BLOC 10 ecrit dans RUN_DIR / "eval_best_dso_v7" : les CSV et le test_summary.json FIGES de la v7
# (sha256 publie, § 8). Pour l'executer tel quel sans les reecrire, RUN_DIR pointe, le temps du BLOC 10,
# vers un dossier LOCAL qui contient une copie de best_dso_v7.pt (sha256 verifie). Le BLOC 10 y charge
# le meme modele et y ecrit ses resultats rejoues, archives ensuite par M1-e dans eval_m1_bloc10i/bloc10_rejoue/.
# RUN_DIR revient au Drive dans M1-d, avant la cellule 10i.
import shutil

assert RUN_DIR == M1_DRIVE_RUN, f"RUN_DIR inattendu : {RUN_DIR}"
M1_LOCAL_RUN = Path("/content/m1_run_v7")
M1_LOCAL_RUN.mkdir(parents=True, exist_ok=True)
_dst = M1_LOCAL_RUN / "best_dso_v7.pt"
t0 = time.time()
if not _dst.exists() or m1_sha256(_dst) != M1_SHA["best_dso_v7.pt"]:
    shutil.copy2(M1_DRIVE_RUN / "best_dso_v7.pt", _dst)
assert m1_sha256(_dst) == M1_SHA_PUBLIE["best_dso_v7.pt"], "copie locale de best_dso_v7.pt corrompue"
RUN_DIR = M1_LOCAL_RUN
print(f"best_dso_v7.pt copie en local ({time.time() - t0:.0f} s, sha256 conforme) | RUN_DIR (BLOC 10) = {RUN_DIR}")
print("M1-c OK")


## BLOC 10 — Evaluation sur le test et DSO

In [ ]:
# BLOC 10 — Evaluation sur le test et DSO
if "ckpt_best" not in globals() or not Path(ckpt_best).exists():
    ckpt_best = RUN_DIR / "best.pt"
_dso_v62 = RUN_DIR / "best_dso_v7.pt"
if _dso_v62.exists():
    ckpt_best = _dso_v62                     # modele retenu par le raffinement v7
ck = torch.load(ckpt_best, map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
EVAL_DIR = RUN_DIR / f"eval_{Path(ckpt_best).stem}"   # un dossier de resultats par modele evalue
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print("Modele evalue :", Path(ckpt_best).name, "| resultats ->", EVAL_DIR)
print(f"Meilleur modele : epoque {ck['epoch'] + 1} (score val = {ck['val_score']:.5f})")

STAMP_OF = dict(zip(INDEX["stem"], INDEX["timestamp"]))
COVER = VALID.reshape(NZ, -1).mean(axis=1)                 # part du domaine au-dessus du relief
EVAL_LEVELS = [k for k in range(NZ) if COVER[k] >= MIN_LEVEL_COVER]

def evaluate(loader, stems):
    """Metriques globales (cumulees sur tous les instants) par niveau et par composante."""
    acc = {c: {k: np.zeros(4) for k in range(NZ)} for c in "uvw"}    # n, sse, sum t, sum t^2
    spd_sum = np.zeros(NZ)
    per_instant = []
    with torch.no_grad():
        for s_raw, sensor, tfeat, y, j in loader:
            s_raw, sensor, tfeat, y = (t.to(device) for t in (s_raw, sensor, tfeat, y))
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                pred = model(assemble_input(s_raw, sensor, tfeat))
            u, v, w = denorm(pred.float())
            row = {"stem": stems[int(j)], "timestamp": STAMP_OF[stems[int(j)]]}
            for c, p in zip("uvw", (u, v, w)):
                t = y[:, "uvw".index(c)]
                for k in range(NZ):
                    m = VALID_T[k]
                    if m.sum() == 0:
                        continue
                    pk, tk = p[0, k][m].double(), t[0, k][m].double()
                    acc[c][k] += np.array([tk.numel(), ((pk - tk) ** 2).sum().item(),
                                           tk.sum().item(), (tk ** 2).sum().item()])
                pv, tv = p[0][VALID_T], t[0][VALID_T]
                row[f"rmse_{c}"] = torch.sqrt(((pv - tv) ** 2).mean()).item()
            for k in range(NZ):
                m = VALID_T[k]
                if m.sum() > 0:
                    spd_sum[k] += torch.hypot(y[0, 0, k][m], y[0, 1, k][m]).double().sum().item()
            dp = torch.rad2deg(torch.atan2(v[0], u[0]))[VALID_T]
            dt_ = torch.rad2deg(torch.atan2(y[0, 1], y[0, 0]))[VALID_T]
            dd = torch.abs(dp - dt_) % 360
            dd = torch.minimum(dd, 360 - dd)
            calm = torch.hypot(y[0, 0], y[0, 1])[VALID_T] < 1.0     # direction non definie par vent calme
            row["mae_dir_deg"] = (dd[~calm].mean() if (~calm).any() else dd.mean()).item()
            r = mass_residual(y[:, 3], u, v, w)
            row["mass_rms"] = torch.sqrt((r[:, MASS_MASK_T] ** 2).mean()).item()

            row["roughness_ratio"] = ((roughness(u) + roughness(v))
                                      / (roughness(y[:, 0]) + roughness(y[:, 1]))).item()
            per_instant.append(row)

    levels = []
    for k in range(NZ):
        rec = {"z_m": int(Z_LEVELS[k]), "couverture": float(COVER[k]),
               "vitesse_moyenne": float(spd_sum[k] / max(acc["u"][k][0], 1))}
        for c in "uvw":
            n, sse, st, st2 = acc[c][k]
            if n == 0:
                continue
            sst = st2 - st ** 2 / n
            rec[f"rmse_{c}"] = float(np.sqrt(sse / n))
            rec[f"r2_{c}"] = float(1 - sse / sst) if sst > 0 else float("nan")
        levels.append(rec)
    return pd.DataFrame(levels), pd.DataFrame(per_instant)

def is_synoptic(stamp):
    t = pd.Timestamp(stamp)
    return t.minute == 0 and t.second == 0 and t.hour % 3 == 0

SYN = np.array([is_synoptic(STAMP_OF[s_]) for s_ in te_s])
syn_idx, ep_idx = np.nonzero(SYN)[0], np.nonzero(~SYN)[0]

def loader_of(idx, seed):
    ds = ANEO3D(te_s[idx], te_h[idx], te_d[idx], train=False, seed=seed)
    return DataLoader(ds, batch_size=1), te_s[idx]

LEVELS, res = evaluate(*loader_of(syn_idx, 2000))                 # DSO1 : instants synoptiques
LEVELS_EP, res_ep = (evaluate(*loader_of(ep_idx, 2500)) if len(ep_idx)
                     else (pd.DataFrame(), pd.DataFrame()))
res["type"] = "synoptique"
if len(res_ep):
    res_ep["type"] = "episode"
LEVELS.to_csv(EVAL_DIR / "test_par_niveau_synoptique.csv", index=False)
if len(LEVELS_EP):
    LEVELS_EP.to_csv(EVAL_DIR / "test_par_niveau_episodes.csv", index=False)
res_all = pd.concat([res, res_ep], ignore_index=True)
res_all.to_csv(EVAL_DIR / "test_par_instant.csv", index=False)

# DSO4 : temps d'inference (assemblage de l'entree inclus)
s_raw, sensor, tfeat, y, _ = next(iter(test_loader))
s_raw, sensor, tfeat = s_raw.to(device), sensor.to(device), tfeat.to(device)
with torch.no_grad():
    for _ in range(3):
        model(assemble_input(s_raw, sensor, tfeat))
    if device.type == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(10 if TEST_MODE else 20):
        t0 = time.perf_counter()
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            model(assemble_input(s_raw, sensor, tfeat))
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
t_inf = float(np.mean(times))

# DSO5 et robustesse : fraction de capteurs visibles (sous-ensemble du test)
robust = []
sub = syn_idx[np.unique(np.linspace(0, len(syn_idx) - 1, min(40, len(syn_idx))).astype(int))]
KEEP_EVAL_SAVE = KEEP_EVAL
for keep in (0.1, 0.25, 0.5, 1.0):
    KEEP_EVAL = keep
    ds = ANEO3D(te_s[sub], te_h[sub], te_d[sub], train=False, seed=3000)
    lv, _ = evaluate(DataLoader(ds, batch_size=1), te_s[sub])
    lv = lv[lv["couverture"] >= MIN_LEVEL_COVER]
    robust.append({"capteurs_visibles": keep, "rmse_u_max": lv["rmse_u"].max(),
                   "rmse_v_max": lv["rmse_v"].max(), "r2_u_min": lv["r2_u"].min(),
                   "r2_v_min": lv["r2_v"].min()})
KEEP_EVAL = KEEP_EVAL_SAVE
robust = pd.DataFrame(robust)
robust.to_csv(EVAL_DIR / "robustesse_capteurs.csv", index=False)

# Bilan des DSO
def ok(b):
    return "CONFORME" if b else "NON CONFORME"

ev = LEVELS[LEVELS["couverture"] >= MIN_LEVEL_COVER]
dso1_rmse = bool((ev[["rmse_u", "rmse_v"]].max(axis=1) <= DSO1_RMSE).all())
dso1_r2 = bool((ev[["r2_u", "r2_v"]].min(axis=1) >= DSO1_R2).all())
r2_all = bool((ev[["r2_u", "r2_v", "r2_w"]].min(axis=1) >= DSO_R2_ALL).all())
# DSO2 et DSO3 sur les instants synoptiques, comme DSO1 (les episodes rapides sont un test de stress)
mass_rms_mean = float(res["mass_rms"].mean())
mass_rms_p95 = float(res["mass_rms"].quantile(0.95))
mass_rms_max = float(res["mass_rms"].max())
rough_mean = float(res["roughness_ratio"].mean())
mass_rms_all = float(res_all["mass_rms"].mean())
rough_all = float(res_all["roughness_ratio"].mean())
r25 = robust[robust["capteurs_visibles"] == 0.25].iloc[0]
dso5 = max(r25["rmse_u_max"], r25["rmse_v_max"]) <= DSO5_RMSE

print("=" * 78)
print(f"TEST SYNOPTIQUE (DSO1) : {len(res)} instants | capteurs visibles : {KEEP_EVAL:.0%}")
print(" Niveau | couverture | RMSE u  RMSE v  RMSE w |  R2 u    R2 v    R2 w")
for _, r in LEVELS.iterrows():
    flag = "" if r["couverture"] >= MIN_LEVEL_COVER else "  (exclu : couverture < 5 %)"
    print(f" {int(r['z_m']):5d} m |   {r['couverture']:5.1%}   | {r['rmse_u']:.3f}   {r['rmse_v']:.3f}   "
          f"{r['rmse_w']:.3f} | {r['r2_u']:.4f}  {r['r2_v']:.4f}  {r['r2_w']:.4f}{flag}")
print("-" * 78)
print(f"DSO1  RMSE u, v <= {DSO1_RMSE} m/s a chaque niveau        -> {ok(dso1_rmse)}")
print(f"      R2 u, v >= {DSO1_R2} a chaque niveau               -> {ok(dso1_r2)}")
print(f"      R2 u, v, w >= {DSO_R2_ALL} a chaque niveau            -> {ok(r2_all)}")
_g = {c: float(np.sqrt((ev[f"rmse_{c}"] ** 2 * ev["couverture"]).sum() / ev["couverture"].sum()))
      for c in "uv"}
dso1_global = max(_g.values()) <= DSO1_RMSE
print(f"      RMSE GLOBAL (tous les noeuds 3D) u = {_g['u']:.3f}  v = {_g['v']:.3f} m/s (seuil {DSO1_RMSE})"
      f" -> {ok(dso1_global)}")
omm = bool((ev[["rmse_u", "rmse_v"]].max(axis=1)
             <= np.maximum(0.5, 0.10 * ev["vitesse_moyenne"])).all())
print(f"      (info) critere OMM : RMSE <= max(0.5 m/s ; 10 % de la vitesse moyenne) -> {ok(omm)}")
print(f"DSO2  RMS residu moyen = {mass_rms_mean:.2e} (seuil {DSO2_RMS:.0e}) -> {ok(mass_rms_mean <= DSO2_RMS)}"
      f" | p95 = {mass_rms_p95:.2e} | max = {mass_rms_max:.2e}")
print(f"      (info) tous instants, episodes compris : RMS residu moyen = {mass_rms_all:.2e}")
print(f"DSO3  rugosite prediction / verite = {rough_mean:.3f} (0.9 - 1.1) -> "
      f"{ok(DSO3_RANGE[0] <= rough_mean <= DSO3_RANGE[1])}")
print(f"      (info) tous instants, episodes compris : rugosite = {rough_all:.3f}")
print(f"DSO4  inference = {t_inf * 1000:.0f} ms (seuil 1 s)              -> {ok(t_inf < DSO4_TIME)}")
print(f"DSO5  RMSE max a 25 % de capteurs = {max(r25['rmse_u_max'], r25['rmse_v_max']):.3f} m/s "
      f"(seuil {DSO5_RMSE}) -> {ok(dso5)}")
print(f"Direction : MAE = {res_all['mae_dir_deg'].mean():.2f} deg (vents >= 1 m/s)")
print("=" * 78)
if len(LEVELS_EP):
    print(f"EPISODES RAPIDES (rafales, front, brise) : {len(res_ep)} instants — test de stress, hors DSO1")
    print(" Niveau | RMSE u  RMSE v |  R2 u    R2 v")
    for _, r in LEVELS_EP[LEVELS_EP["couverture"] >= MIN_LEVEL_COVER].iterrows():
        print(f" {int(r['z_m']):5d} m | {r['rmse_u']:.3f}   {r['rmse_v']:.3f} | {r['r2_u']:.4f}  {r['r2_v']:.4f}")
    print("=" * 78)
print("Detail par type d'instant :")
for _name, _d in (("synoptique", res), ("episodes", res_ep)):
    if len(_d):
        print(f"  {_name:11s} | masse RMS moy = {_d['mass_rms'].mean():.2e} | p95 = {_d['mass_rms'].quantile(0.95):.2e}"
              f" | rugosite = {_d['roughness_ratio'].mean():.3f} | MAE dir = {_d['mae_dir_deg'].mean():.2f} deg")
print("=" * 78)
print("Robustesse (instants synoptiques repartis sur tous les jours de test) :")
print(robust.round(4).to_string(index=False))

summary = {"instants_synoptiques": len(res), "instants_episodes": len(res_ep),
           "capteurs_visibles": KEEP_EVAL,
           "par_niveau_synoptique": LEVELS.to_dict(orient="records"),
           "par_niveau_episodes": LEVELS_EP.to_dict(orient="records") if len(LEVELS_EP) else [],
           "dso1_rmse": dso1_rmse, "dso1_rmse_global_u": _g["u"], "dso1_rmse_global_v": _g["v"], "dso1_r2": dso1_r2, "dso2_rms_moyen": mass_rms_mean,
           "dso2_rms_p95": mass_rms_p95, "dso2_rms_max": mass_rms_max,
           "dso2_rms_tous_instants": mass_rms_all, "dso3_rugosite_tous_instants": rough_all,
           "modele": Path(ckpt_best).name,
           "dso3_rugosite": rough_mean, "dso4_inference_s": t_inf,
           "dso5_rmse_25pct": float(max(r25["rmse_u_max"], r25["rmse_v_max"])),
           "mae_direction_deg": float(res_all["mae_dir_deg"].mean())}
with open(EVAL_DIR / "test_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("BLOC 10 OK")

## M1-d — Sorties de 10i dans eval_m1_bloc10i/

In [ ]:
# M1-d — Avant la cellule 10i : RUN_DIR revient au Drive (10i lit RUN_DIR / "couche_limite_v7.json", fige,
# sans l'ecrire) et EVAL_DIR pointe vers eval_m1_bloc10i/ (§ 5.1 point 8) : 10i y ecrit
# decision_10i_par_niveau.csv et decision_10i.json, jamais dans eval_best_dso_v7/.
M1_BLOC10_DIR = Path(EVAL_DIR)                       # resultats rejoues du BLOC 10 (disque local)
assert str(M1_BLOC10_DIR).startswith(str(M1_LOCAL_RUN)), f"le BLOC 10 a ecrit hors du dossier local : {M1_BLOC10_DIR}"
RUN_DIR = M1_DRIVE_RUN
assert m1_sha256(RUN_DIR / "couche_limite_v7.json") == M1_SHA["couche_limite_v7.json"], "couche_limite_v7.json a change"
if (M1_EVAL_DIR / "decision_10i.json").exists():
    raise RuntimeError("decision_10i.json existe deja : regle d'execution unique (§ 5.2)")
M1_EVAL_DIR.mkdir(parents=True, exist_ok=True)
_restes = [p for p in M1_EVAL_DIR.iterdir() if not p.name.startswith("_tentative_interrompue_")]
if _restes:                                          # tentative interrompue (§ 5.2) : gardee a part, jamais effacee
    _dest = M1_EVAL_DIR / f"_tentative_interrompue_{time.strftime('%Y%m%d_%H%M%S')}"
    _dest.mkdir()
    for p in _restes:
        shutil.move(str(p), str(_dest / p.name))
    print(f"Fichiers d'une tentative interrompue deplaces dans {_dest.name} : {[p.name for p in _restes]}")
EVAL_DIR = M1_EVAL_DIR
print(f"RUN_DIR = {RUN_DIR} | EVAL_DIR (10i) = {EVAL_DIR}")
print(f"Cellule suivante : BLOC 10i, code fige, sha256 publie {M1_SHA_10I}")
print("M1-d OK")


## BLOC 10i — Mesure décisive (code figé)

In [ ]:
# ==== CELLULE 10i du notebook fno3d_v7.ipynb (lot M1) — a coller juste APRES le BLOC 10h ====
# SHA-256 du CODE DE LA CELLULE (tout ce qui suit la ligne '# ==== DEBUT', en UTF-8, fins de ligne LF,
# terminant par un seul saut de ligne) :
#   a1fcc878ed3c6c6047538e5b48df25c6633c972a413d6bd42f415926599fbd6f
# Lit seulement : globales des BLOCs 0 a 10 + RUN_DIR/couche_limite_v7.json
#   (Drive : /content/drive/MyDrive/aneo_fno3d_runs/v7/couche_limite_v7.json, fige, jamais reecrit).
# Ecrit seulement : EVAL_DIR/decision_10i_par_niveau.csv et EVAL_DIR/decision_10i.json.
# ==== DEBUT
# BLOC 10i — Evaluation FIGEE de la correction : mesure decisive (lot M1)
# Methode figee : v7 + correction du vent moyen par niveau du BLOC 10h (alpha 0.01, choisi sur la
# VALIDATION), rechargee depuis RUN_DIR/couche_limite_v7.json (meme calcul que le BLOC 12, etape 2b).
# Aucun reglage dans cette cellule : on mesure seulement.
#
# Variantes deja vues sur le TEST pendant la conception (l'estimation du test est donc optimiste) :
#   BLOC 10e (4 variantes, choix fait sur le test), 10f, 10g, 10h alpha 1, 10h alpha 0.01 (grille
#   etendue apres lecture du test de 10h alpha 1). Les 46 episodes n'ont servi a AUCUN choix.
#
# REGLE DE DECISION (fixee avant execution) : correction RETENUE si C1 ET C2 ET C3.
#   C1  test synoptique (DSO1, 50 % des capteurs) : pire niveau u et v apres correction <= 0.5 m/s et
#       R2 >= 0.95 a chaque niveau ; l'IC 95 % bootstrap est rapporte (ce n'est pas un seuil).
#   C2  episodes (46 instants jamais utilises) : la correction ne degrade pas.
#       Delta = RMSE apres - RMSE avant, bootstrap apparie (1000 tirages), u et v :
#       650-1000 m : borne haute de l'IC 95 % de Delta <= 0       (strict)
#       100-500 m  : borne haute de l'IC 95 % de Delta <= +0.02   (tolerant)
#   C3  DSO2, DSO3, DSO5 conformes et DSO4 (modele + correction) < 1 s.
#   Information, hors regle : validation avant/apres par niveau, test synoptique a 100 % des capteurs.
# BOOTSTRAP (choisi par Rayen avant execution) fixe l'IC utilise par C2 ; l'autre est rapporte pour information.
#   "blocs"    : episodes -> blocs de BLOC_LEN instants consecutifs, reechantillonnes A L'INTERIEUR de chaque
#                episode (les 46 instants viennent de 3 situations : les instants voisins sont correles) ;
#                test synoptique -> jours entiers reechantillonnes (grappes de 8 instants).
#                NB : avec 3 episodes seulement, l'IC est conditionnel a ces 3 situations.
#   "instants" : tirage simple des instants avec remise (IC trop etroit si les instants sont correles).
# Ecrit : EVAL_DIR/decision_10i_par_niveau.csv et EVAL_DIR/decision_10i.json (nouveaux fichiers).
import hashlib

BOOTSTRAP = "blocs"      # "blocs" | "instants" : IC utilise par C2 -- decide par Rayen (salon #332, avenant 33da965)
BLOC_LEN = 6             # longueur des blocs d'instants consecutifs (mode "blocs", episodes)
assert BOOTSTRAP in ("blocs", "instants")
for _v in ("syn_idx", "ep_idx", "EVAL_LEVELS", "COVER", "EVAL_DIR", "amp_dtype", "vsel"):
    if _v not in globals():
        raise RuntimeError("Executer d'abord le BLOC 9 puis le BLOC 10.")
assert KEEP_EVAL == 0.5, "KEEP_EVAL doit valoir 0.5 (protocole DSO1) : relancer le BLOC 1"
assert len(ep_idx) > 0, "aucun instant d'episode dans le test"
CORR_PATH = RUN_DIR / "couche_limite_v7.json"
with open(CORR_PATH, encoding="utf-8") as f:
    CORR = json.load(f)
assert abs(CORR["ridge_alpha"] - 0.01) < 1e-12, "couche_limite_v7.json doit venir du BLOC 10h avec alpha 0.01"
CORR_SHA = hashlib.sha256(CORR_PATH.read_bytes()).hexdigest()
C_PAR = {key: (np.array(p["mu"]), np.array(p["sd"]), np.array(p["coef"]))
         for key, p in CORR["niveaux"].items() if p["utilise"]}
model.eval()

print("=" * 96)
print("REGLE DE DECISION (fixee avant execution) : correction retenue si C1 et C2 et C3")
print("  C1  test synoptique : pire niveau u et v <= 0.5 m/s et R2 >= 0.95 a chaque niveau (IC 95 % rapporte)")
print("  C2  episodes : borne haute IC 95 % de (apres - avant) <= 0 a 650-1000 m, <= +0.02 a 100-500 m (u et v)")
print("  C3  DSO2, DSO3, DSO5 conformes et DSO4 (modele + correction) < 1 s")
print(f"IC pour C2 : BOOTSTRAP = '{BOOTSTRAP}' (blocs de {BLOC_LEN} instants a l'interieur de chaque episode ; "
      f"test : jours entiers) | l'autre mode est rapporte pour information")
print(f"Correction : {CORR_PATH.name} | alpha {CORR['ridge_alpha']:g} | rho_sl {CORR['variante_rho_sl']} "
      f"| {len(C_PAR)} niveaux x composantes | sha256 {CORR_SHA[:16]}")
print("=" * 96)


def correction_targets(S, sensor, tfeat, um, vm):
    """Moyenne cible de u, v par niveau : dict (k, c) -> m/s. Meme calcul que le BLOC 12, etape 2b.
    S (>= 7, NY, NX), sensor booleen, tfeat (8,), um / vm : moyennes par niveau predites par le modele."""
    m_s = sensor
    p_s = S[6][m_s].astype(np.float64)
    tv_s = p_s / (287.05 * S[4][m_s].astype(np.float64))      # temperature virtuelle (gaz parfait)
    z_s = Z_SENSOR[m_s]
    p_sl = p_s * np.exp(9.80665 * z_s / (287.05 * (tv_s + 0.5 * 0.0065 * z_s)))
    A_s = np.stack([np.ones(m_s.sum()), XC[m_s], YC[m_s], z_s], axis=1)
    cf, *_ = np.linalg.lstsq(A_s, p_sl, rcond=None)
    res_s = p_sl - A_s @ cf
    mad = 1.4826 * np.median(np.abs(res_s - np.median(res_s))) + 1e-9
    keep_s = np.abs(res_s - np.median(res_s)) < 5.0 * mad
    if keep_s.sum() > 100:
        cf, *_ = np.linalg.lstsq(A_s[keep_s], p_sl[keep_s], rcond=None)
    rho_sl = 1.2 if CORR["variante_rho_sl"] == "1.2" else float(np.mean(p_sl / (287.05 * (tv_s + 0.0065 * z_s))))
    ug2, vg2 = -cf[2] / (rho_sl * F_CORIOLIS), cf[1] / (rho_sl * F_CORIOLIS)
    d = cf[3] * 1e3                                          # indicateur de stabilite (Pa/km)
    surf_m = [float(S[c][m_s].mean()) for c in (0, 1, 3)]    # u, v, T moyens des capteurs visibles
    a = math.atan2(surf_m[1], surf_m[0]) - math.atan2(vg2, ug2)
    a = (a + math.pi) % (2 * math.pi) - math.pi              # angle de rotation estime
    rot = []
    for fr in (0.25, 0.5, 0.75):
        rot += [(ug2 * math.cos(fr * a) - vg2 * math.sin(fr * a)) / 10.0,
                (ug2 * math.sin(fr * a) + vg2 * math.cos(fr * a)) / 10.0]
    su, sv = float(tfeat[6]), float(tfeat[7])                 # vent thermique (x 200)
    tf = list(np.asarray(tfeat, dtype=np.float64))
    out = {}
    for k in range(NZ):
        for c in "uv":
            par = C_PAR.get(f"{int(Z_LEVELS[k])}_{c}")
            if par is None:
                continue
            x = np.array(tf + surf_m + [um[k], vm[k]] + [ug2 / 10.0, vg2 / 10.0, d, su * d, sv * d] + rot
                         + [su * max(d, 0.0), su * min(d, 0.0), sv * max(d, 0.0), sv * min(d, 0.0),
                            ug2 / 10.0 * d, vg2 / 10.0 * d])
            out[(k, c)] = float(np.r_[1.0, (x - par[0]) / par[1]] @ par[2])
    return out


def predict_corrected(s_raw, sensor, tfeat):
    """(u, v, w) du modele seul et (u, v) corriges ; tenseurs (1, NZ, NY, NX) sur le GPU."""
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        pred = model(assemble_input(s_raw.to(device), sensor.to(device), tfeat.to(device)))
    u, v, w = denorm(pred.float())
    um = np.array([u[0, k][VALID_T[k]].mean().item() if VALID[k].any() else 0.0 for k in range(NZ)])
    vm = np.array([v[0, k][VALID_T[k]].mean().item() if VALID[k].any() else 0.0 for k in range(NZ)])
    tg = correction_targets(s_raw[0].numpy(), sensor[0].numpy() > 0.5, tfeat[0].numpy(), um, vm)
    uc, vc = u.clone(), v.clone()
    for (k, c), t_ in tg.items():
        f_ = uc if c == "u" else vc
        mk = VALID_T[k]
        f_[0, k] += (t_ - f_[0, k][mk].mean()) * mk          # decalage des noeuds valides du niveau
    return u, v, w, uc, vc


def run(ds):
    """Par instant : SSE avant / apres (N, NZ, 2), sommes de la verite (R2), masse et rugosite apres."""
    N = len(ds)
    R = {"avant": np.zeros((N, NZ, 2)), "apres": np.zeros((N, NZ, 2)),
         "st": np.zeros((N, NZ, 2)), "st2": np.zeros((N, NZ, 2)), "mass": [], "rough": []}
    for i, (s_raw, sensor, tfeat, y, _) in enumerate(DataLoader(ds, batch_size=1, num_workers=NUM_WORKERS)):
        u, v, w, uc, vc = predict_corrected(s_raw, sensor, tfeat)
        y = y.to(device)
        for k in EVAL_LEVELS:
            mk = VALID_T[k]
            for ci in range(2):
                tk = y[0, ci, k][mk].double()
                R["avant"][i, k, ci] = (((u, v)[ci][0, k][mk].double() - tk) ** 2).sum().item()
                R["apres"][i, k, ci] = (((uc, vc)[ci][0, k][mk].double() - tk) ** 2).sum().item()
                R["st"][i, k, ci], R["st2"][i, k, ci] = tk.sum().item(), (tk ** 2).sum().item()
        r = mass_residual(y[:, 3], uc, vc, w)
        R["mass"].append(torch.sqrt((r[:, MASS_MASK_T] ** 2).mean()).item())
        R["rough"].append(((roughness(uc) + roughness(vc)) / (roughness(y[:, 0]) + roughness(y[:, 1]))).item())
    return R


def run_keep(stems_idx, seed, keep, stems=None, hours=None, doys=None):
    global KEEP_EVAL
    saved = KEEP_EVAL
    try:
        KEEP_EVAL = keep
        st_, hr_, dy_ = (te_s, te_h, te_d) if stems is None else (stems, hours, doys)
        return run(ANEO3D(st_[stems_idx], hr_[stems_idx], dy_[stems_idx], train=False, seed=seed))
    finally:
        KEEP_EVAL = saved


NK = np.maximum(VALID.reshape(NZ, -1).sum(axis=1), 1).astype(np.float64)
LV = np.array(EVAL_LEVELS)
B = 1000


def rmse_lv(sse_sum, n_inst):
    return np.sqrt(sse_sum / (n_inst * NK[:, None]))              # (NZ, 2)


DAY_OF = dict(zip(INDEX["stem"], INDEX["day"]))


def sampler(stems, mode, kind, seed):
    """Fonction qui tire un vecteur d'indices d'instants (avec remise).
    mode "instants" : tirage simple. mode "blocs" : kind "episodes" -> blocs de BLOC_LEN instants consecutifs
    reechantillonnes a l'interieur de chaque episode (jour) ; kind "jours" -> jours entiers reechantillonnes."""
    rng = np.random.default_rng(seed)
    n_all = len(stems)
    days = np.array([DAY_OF[s] for s in stems])
    groups = [np.nonzero(days == d)[0] for d in pd.unique(days)]   # instants dans l'ordre chronologique
    if mode == "instants":
        return lambda: rng.integers(0, n_all, n_all)
    if kind == "jours":
        return lambda: np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])

    def draw():
        out = []
        for g in groups:
            n, L = len(g), min(BLOC_LEN, len(g))
            starts = rng.integers(0, n - L + 1, math.ceil(n / L))
            out.append(np.concatenate([g[s:s + L] for s in starts])[:n])
        return np.concatenate(out)
    return draw


def boot(R, draw):
    """RMSE complets et tirages bootstrap apparies (memes instants avant / apres)."""
    N = R["avant"].shape[0]
    full = {n: rmse_lv(R[n].sum(0), N) for n in ("avant", "apres")}
    dr = {n: np.empty((B, NZ, 2)) for n in ("avant", "apres")}
    for b in range(B):
        idx = draw()
        for n in ("avant", "apres"):
            dr[n][b] = rmse_lv(R[n][idx].sum(0), len(idx))
    return full, dr


def ci(x):
    return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))


def worst(full, dr, n, ci_):
    return float(full[n][LV, ci_].max()), ci(dr[n][:, LV, ci_].max(axis=1))


# ---------------------------------------------------------------- 1. Mesures
t0 = time.time()
R_val = run(ANEO3D(va_s[vsel], va_h[vsel], va_d[vsel], train=False, seed=1000))
R_syn = run(ANEO3D(te_s[syn_idx], te_h[syn_idx], te_d[syn_idx], train=False, seed=2000))
R_ep = run(ANEO3D(te_s[ep_idx], te_h[ep_idx], te_d[ep_idx], train=False, seed=2500))
R_100 = run_keep(syn_idx, 2000, 1.0)
sub = syn_idx[np.unique(np.linspace(0, len(syn_idx) - 1, min(40, len(syn_idx))).astype(int))]
R_25 = run_keep(sub, 3000, 0.25)
print(f"Instants : validation {len(vsel)} | test synoptique {len(syn_idx)} | episodes {len(ep_idx)} "
      f"| 100 % capteurs {len(syn_idx)} | 25 % capteurs {len(sub)} ({time.time() - t0:.0f} s)")
BOOTS = {}
for mode in ("instants", "blocs"):
    BOOTS[("syn", mode)] = boot(R_syn, sampler(te_s[syn_idx], mode, "jours", 101))
    BOOTS[("ep", mode)] = boot(R_ep, sampler(te_s[ep_idx], mode, "episodes", 202))
F_syn, D_syn = BOOTS[("syn", BOOTSTRAP)]
F_ep, D_ep = BOOTS[("ep", BOOTSTRAP)]
OTHER = "instants" if BOOTSTRAP == "blocs" else "blocs"
D_syn_o, D_ep_o = BOOTS[("syn", OTHER)][1], BOOTS[("ep", OTHER)][1]
_ep_days = pd.Series([DAY_OF[s] for s in te_s[ep_idx]]).value_counts(sort=False)
print("Episodes par jour : " + ", ".join(f"jour {d} : {n} instants" for d, n in _ep_days.items()))
F_val = {n: rmse_lv(R_val[n].sum(0), R_val["avant"].shape[0]) for n in ("avant", "apres")}
F_100 = {n: rmse_lv(R_100[n].sum(0), R_100["avant"].shape[0]) for n in ("avant", "apres")}
F_25 = {n: rmse_lv(R_25[n].sum(0), R_25["avant"].shape[0]) for n in ("avant", "apres")}

# DSO4 : modele seul / modele + correction (assemblage de l'entree inclus, comme au BLOC 10)
s1, m1, t1_, _, _ = ANEO3D(te_s[syn_idx], te_h[syn_idx], te_d[syn_idx], train=False, seed=2000)[0]
s1, m1, t1_ = s1[None], m1[None], t1_[None]


def timed(correct):
    t_ = time.perf_counter()
    if correct:
        predict_corrected(s1, m1, t1_)
    else:
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            denorm(model(assemble_input(s1.to(device), m1.to(device), t1_.to(device))).float())
    if device.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t_


for _ in range(3):
    timed(True)
T_INF = {c: float(np.mean([timed(c) for _ in range(10 if TEST_MODE else 20)])) for c in (False, True)}

# ---------------------------------------------------------------- 2. Tableaux
rows = []
print("=" * 96)
print("VALIDATION (information) : RMSE par niveau, avant -> apres")
print(" Niveau |  u avant  u apres |  v avant  v apres")
for k in LV:
    print(f" {int(Z_LEVELS[k]):5d} m |   {F_val['avant'][k, 0]:.3f}   {F_val['apres'][k, 0]:.3f}  |"
          f"   {F_val['avant'][k, 1]:.3f}   {F_val['apres'][k, 1]:.3f}")
print("=" * 96)
print(f"TEST SYNOPTIQUE (C1) : {len(syn_idx)} instants, 50 % des capteurs")
print(" Niveau |  u avant  u apres  [IC 95 % apres] |  v avant  v apres  [IC 95 % apres]")
for k in LV:
    cu, cv = ci(D_syn["apres"][:, k, 0]), ci(D_syn["apres"][:, k, 1])
    print(f" {int(Z_LEVELS[k]):5d} m |   {F_syn['avant'][k, 0]:.3f}   {F_syn['apres'][k, 0]:.3f}  [{cu[0]:.3f}, {cu[1]:.3f}]"
          f"  |   {F_syn['avant'][k, 1]:.3f}   {F_syn['apres'][k, 1]:.3f}  [{cv[0]:.3f}, {cv[1]:.3f}]")
print("(controle : 'avant' = tableau du BLOC 10 ; 'apres' = colonne 'couche limite' du BLOC 10h, alpha 0.01)")
print("=" * 96)
print(f"EPISODES (C2) : {len(ep_idx)} instants jamais utilises, 50 % des capteurs | IC : bootstrap '{BOOTSTRAP}'"
      f" (borne haute en mode '{OTHER}' entre parentheses, information)")
print(" Niveau |  u avant  u apres  IC 95 % (apres-avant)          |  v avant  v apres  IC 95 % (apres-avant)          | seuil | C2")
c2 = True
for k in LV:
    tol = 0.0 if Z_LEVELS[k] >= 650 else 0.02
    d_u = ci(D_ep["apres"][:, k, 0] - D_ep["avant"][:, k, 0])
    d_v = ci(D_ep["apres"][:, k, 1] - D_ep["avant"][:, k, 1])
    o_u = ci(D_ep_o["apres"][:, k, 0] - D_ep_o["avant"][:, k, 0])[1]
    o_v = ci(D_ep_o["apres"][:, k, 1] - D_ep_o["avant"][:, k, 1])[1]
    ok_k = d_u[1] <= tol + 1e-12 and d_v[1] <= tol + 1e-12
    c2 = c2 and ok_k
    print(f" {int(Z_LEVELS[k]):5d} m |   {F_ep['avant'][k, 0]:.3f}   {F_ep['apres'][k, 0]:.3f}  [{d_u[0]:+.3f}, {d_u[1]:+.3f}] ({o_u:+.3f})"
          f"  |   {F_ep['avant'][k, 1]:.3f}   {F_ep['apres'][k, 1]:.3f}  [{d_v[0]:+.3f}, {d_v[1]:+.3f}] ({o_v:+.3f})"
          f"  | {tol:+.2f} | {'OK' if ok_k else 'NON'}")
print("(controle : 'avant' = tableau EPISODES RAPIDES du BLOC 10)")
for k in LV:
    for ci_, c in enumerate("uv"):
        rows.append({"z_m": int(Z_LEVELS[k]), "comp": c,
                     "val_avant": F_val["avant"][k, ci_], "val_apres": F_val["apres"][k, ci_],
                     "syn_avant": F_syn["avant"][k, ci_], "syn_apres": F_syn["apres"][k, ci_],
                     "syn_apres_ic_bas": ci(D_syn["apres"][:, k, ci_])[0],
                     "syn_apres_ic_haut": ci(D_syn["apres"][:, k, ci_])[1],
                     "ep_avant": F_ep["avant"][k, ci_], "ep_apres": F_ep["apres"][k, ci_],
                     "ep_delta_ic_bas": ci(D_ep["apres"][:, k, ci_] - D_ep["avant"][:, k, ci_])[0],
                     "ep_delta_ic_haut": ci(D_ep["apres"][:, k, ci_] - D_ep["avant"][:, k, ci_])[1],
                     "ep_delta_ic_haut_autre_mode": ci(D_ep_o["apres"][:, k, ci_] - D_ep_o["avant"][:, k, ci_])[1],
                     "bootstrap": BOOTSTRAP,
                     "cap100_avant": F_100["avant"][k, ci_], "cap100_apres": F_100["apres"][k, ci_],
                     "cap25_avant": F_25["avant"][k, ci_], "cap25_apres": F_25["apres"][k, ci_]})
pd.DataFrame(rows).to_csv(EVAL_DIR / "decision_10i_par_niveau.csv", index=False)

# ---------------------------------------------------------------- 3. Pires niveaux, criteres, verdict
def ok(b):
    return "CONFORME" if b else "NON CONFORME"


N_syn = R_syn["avant"].shape[0]
n_all = N_syn * NK[:, None]
r2_apres = 1 - R_syn["apres"].sum(0) / (R_syn["st2"].sum(0) - R_syn["st"].sum(0) ** 2 / n_all)
W = {}
print("=" * 96)
print(f"PIRE NIVEAU (IC 95 %, 1000 tirages ; mode '{BOOTSTRAP}', puis mode '{OTHER}' pour information)")
for lab, F_, D_, Do_ in (("test synoptique", F_syn, D_syn, D_syn_o), ("episodes", F_ep, D_ep, D_ep_o)):
    for ci_, c in enumerate("uv"):
        wa, ca = worst(F_, D_, "avant", ci_)
        wp, cp = worst(F_, D_, "apres", ci_)
        _, cao = worst(F_, Do_, "avant", ci_)
        _, cpo = worst(F_, Do_, "apres", ci_)
        W[f"{lab}_{c}"] = {"avant": wa, "avant_ic95": ca, "apres": wp, "apres_ic95": cp,
                           "avant_ic95_autre_mode": cao, "apres_ic95_autre_mode": cpo}
        print(f"  {lab:15s} {c} : avant {wa:.3f} [{ca[0]:.3f}, {ca[1]:.3f}] -> apres {wp:.3f} [{cp[0]:.3f}, {cp[1]:.3f}]"
              f" m/s   ({OTHER} : [{cao[0]:.3f}, {cao[1]:.3f}] -> [{cpo[0]:.3f}, {cpo[1]:.3f}])")
w100 = {c: (float(F_100["avant"][LV, ci_].max()), float(F_100["apres"][LV, ci_].max())) for ci_, c in enumerate("uv")}
print(f"  100 % capteurs (information) : u {w100['u'][0]:.3f} -> {w100['u'][1]:.3f} | v {w100['v'][0]:.3f} -> {w100['v'][1]:.3f} m/s")
c1 = (W["test synoptique_u"]["apres"] <= DSO1_RMSE and W["test synoptique_v"]["apres"] <= DSO1_RMSE
      and bool((r2_apres[LV] >= DSO1_R2).all()))
m_c, r_c = float(np.mean(R_syn["mass"])), float(np.mean(R_syn["rough"]))
r5 = float(F_25["apres"][LV].max())
c3 = (m_c <= DSO2_RMS) and (0.9 <= r_c <= 1.1) and (r5 <= DSO5_RMSE) and (T_INF[True] < DSO4_TIME)
print("=" * 96)
print(f"C1  DSO1 sur le test synoptique (pire niveau <= {DSO1_RMSE}, R2 min {r2_apres[LV].min():.4f})  -> {ok(c1)}")
print(f"C2  episodes : la correction ne degrade pas (650-1000 m strict, 100-500 m +0.02)        -> {ok(c2)}")
print(f"C3  DSO2 {m_c:.2e} | DSO3 {r_c:.3f} | DSO5 {r5:.3f} | DSO4 {T_INF[True] * 1000:.0f} ms "
      f"(modele seul {T_INF[False] * 1000:.0f} ms)  -> {ok(c3)}")
retained = bool(c1 and c2 and c3)
print("=" * 96)
print("DECISION : correction " + ("RETENUE (v7 + correction)" if retained else "NON RETENUE (v7 seul)"))
print("Reserve : coefficients appris sur donnees synthetiques ; sur donnees reelles, l'erreur des barometres")
print("(+-10 a 50 Pa) est du meme ordre que le signal geostrophique (~26 Pa pour 10 m/s) : a recalibrer.")
with open(EVAL_DIR / "decision_10i.json", "w", encoding="utf-8") as f:
    json.dump({"correction": CORR_PATH.name, "correction_sha256": CORR_SHA, "alpha": CORR["ridge_alpha"],
               "regle": {"C1": "pire niveau u, v <= 0.5 et R2 >= 0.95 (test synoptique, 50 %)",
                         "C2": "IC95 haut de (apres - avant) <= 0 a 650-1000 m, <= +0.02 a 100-500 m (episodes)",
                         "C3": "DSO2, DSO3, DSO5 conformes et DSO4 modele + correction < 1 s"},
               "bootstrap": {"mode_C2": BOOTSTRAP, "bloc_instants": BLOC_LEN, "tirages": B,
                             "episodes_par_jour": {str(d): int(n) for d, n in _ep_days.items()}},
               "pire_niveau": W, "cap100_pire": w100, "r2_min_test_apres": float(r2_apres[LV].min()),
               "dso2": m_c, "dso3": r_c, "dso5_25pct": r5,
               "dso4_s": {"modele": T_INF[False], "modele_plus_correction": T_INF[True]},
               "C1": bool(c1), "C2": bool(c2), "C3": bool(c3), "retenue": retained,
               "instants": {"validation": int(len(vsel)), "test_synoptique": int(len(syn_idx)),
                            "episodes": int(len(ep_idx)), "cap25": int(len(sub))}}, f, indent=2)
print("Resultats :", EVAL_DIR / "decision_10i_par_niveau.csv", "et", EVAL_DIR / "decision_10i.json")
print("BLOC 10i OK")


## M1-e — Archivage et contrôles de fin

In [ ]:
# M1-e — Archivage (§ 5.1 point 8) dans eval_m1_bloc10i/ : sha256.txt, regle.txt, resultats rejoues du BLOC 10.
# Controle le sha256 du code 10i REELLEMENT execute (historique IPython) et verifie que les fichiers figes
# n'ont pas change pendant la session. N'ecrit que dans eval_m1_bloc10i/.
assert EVAL_DIR == M1_EVAL_DIR and (EVAL_DIR / "decision_10i.json").exists(), "la cellule 10i n'a pas termine"
_marque_debut, _marque_fin = "# ==== " + "DEBUT\n", "BLOC 10i" + " OK"
_hist = [c for c in globals().get("In", []) if _marque_debut in c and _marque_fin in c]
if _hist:
    _code = _hist[-1].split(_marque_debut, 1)[1]
    M1_SHA_10I_EXEC = hashlib.sha256((_code.rstrip("\n") + "\n").encode("utf-8")).hexdigest()
else:
    M1_SHA_10I_EXEC = "non disponible (historique IPython absent)"
_ok10i = M1_SHA_10I_EXEC == M1_SHA_10I
t0 = time.time()
_apres = {n: m1_sha256(M1_DRIVE_RUN / n) for n in M1_SHA}
_change = [n for n in M1_SHA if _apres[n] != M1_SHA[n]]
_bloc10 = EVAL_DIR / "bloc10_rejoue"
_bloc10.mkdir(exist_ok=True)
for p in sorted(M1_BLOC10_DIR.iterdir()):
    if p.is_file():
        shutil.copy2(p, _bloc10 / p.name)
with open(EVAL_DIR / "sha256.txt", "w", encoding="utf-8") as f:
    f.write(f"# Session M1, BLOC 10i | debut {M1_DEBUT} | GPU {M1_GPU}\n")
    f.write("# fichiers figes (Drive aneo_fno3d_runs/v7/), sha256 en debut de session :\n")
    for n, s in M1_SHA.items():
        f.write(f"{s}  {n}\n")
    f.write(f"# code de la cellule 10i : publie {M1_SHA_10I} | execute {M1_SHA_10I_EXEC}\n")
    f.write(f"# fichiers figes modifies pendant la session : {_change if _change else 'aucun'}\n")
    for p in sorted(EVAL_DIR.glob("decision_10i*")):
        f.write(f"{m1_sha256(p)}  eval_m1_bloc10i/{p.name}\n")
with open(EVAL_DIR / "regle.txt", "w", encoding="utf-8") as f:
    f.write(M1_REGLE + "\n")
print(f"Code 10i execute : {M1_SHA_10I_EXEC} -> {'CONFORME au sha256 publie' if _ok10i else 'NON CONFORME / non verifie : a signaler'}")
print(f"Fichiers figes inchanges pendant la session : {'oui' if not _change else 'NON : ' + str(_change)} ({time.time() - t0:.0f} s)")
print("Archive :", EVAL_DIR)
for p in sorted(EVAL_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(EVAL_DIR)}  ({p.stat().st_size:,} o)")
print("M1-e OK")


## M1-f — Jeu vierge graine 2027 (avenant n° 2, hors règle)

In [ ]:
# M1-f — Jeu VIERGE graine 2027 (avenant n 2) : mesure HORS REGLE, pour information. N'entre pas dans C1/C2/C3.
# Ne s'execute que si le jeu complet est sur le Drive (M1_VIERGE_DIR, 15 jours, aucun fichier .part) ;
# sinon, message et rien d'autre. Une seule execution : refuse si VIERGE_OUT/summary.json existe.
# Protocole de l'avenant n 2 : instants synoptiques, 50 % des capteurs, graine 2000, niveaux evalues de 10i
# (min_level_cover 0.05), v7 seul et v7 + correction figee (fonctions de la cellule 10i : predict_corrected,
# sampler/boot non utilises pour les grappes, rmse_lv, ci, worst), DSO2, DSO3, DSO5 (25 %, 40 instants,
# graine 3000), IC 95 % par bootstrap sur les JOURS entiers (1000 tirages, graine fixe).
# Ecrit seulement : cache local /content/aneo_cache_seed2027 et EVAL_DIR / "jeu_vierge_seed2027" / *.
M1_VIERGE_DIR = Path("/content/drive/MyDrive/aneo_dataset_3d_seed2027")
M1_VIERGE_CACHE = Path("/content/aneo_cache_seed2027")
M1_VIERGE_OUT = M1_EVAL_DIR / "jeu_vierge_seed2027"
M1_VIERGE_JOURS = 15


def m1_vierge_absent():
    if not (M1_VIERGE_DIR / "meta.json").exists():
        return f"{M1_VIERGE_DIR}/meta.json introuvable : le jeu vierge n'est pas sur le Drive"
    if not (M1_VIERGE_DIR / "aneo_grid_points.csv").exists():
        return f"{M1_VIERGE_DIR}/aneo_grid_points.csv introuvable"
    fichiers = sorted(M1_VIERGE_DIR.glob("aneo_observations_day_*_multitime.csv*"))
    partiels = [p.name for p in fichiers if p.name.endswith(".part")]
    complets = [p for p in fichiers if not p.name.endswith(".part")]
    if partiels:
        return f"fichiers incomplets (.part) : {partiels}"
    if len(complets) != M1_VIERGE_JOURS:
        return f"{len(complets)} jours sur le Drive au lieu de {M1_VIERGE_JOURS}"
    return None


def m1_vierge_grille():
    """Meme geometrie que le jeu v2 (BLOCS 1-2) : meta.json et aneo_grid_points.csv compares."""
    with open(M1_VIERGE_DIR / "meta.json", encoding="utf-8") as f:
        meta2 = json.load(f)
    for key in ("grille_surface", "noeuds_3d"):
        assert meta2[key] == META[key], f"meta.json : '{key}' differe du jeu v2"
    assert meta2["calendrier"]["jours"] == M1_VIERGE_JOURS, meta2["calendrier"]
    g2 = pd.read_csv(M1_VIERGE_DIR / "aneo_grid_points.csv",
                     usecols=["point_id", "ix", "iy", "z_terrain_m", "z_rel_m", "z0_m", "slope_deg", "aspect_deg"])
    s2 = g2[g2["point_id"] < NODE_ID_START].sort_values("point_id").reset_index(drop=True)
    n2 = g2[g2["point_id"] >= NODE_ID_START].sort_values("point_id").reset_index(drop=True)
    s1 = surf.sort_values("point_id").reset_index(drop=True)
    n1 = nodes.sort_values("point_id").reset_index(drop=True)
    for a, b, nom in ((s1, s2, "surface"), (n1, n2, "noeuds 3D")):
        assert len(a) == len(b), f"grille {nom} : {len(b)} points au lieu de {len(a)}"
        for c in ("point_id", "ix", "iy"):
            assert np.array_equal(a[c].values, b[c].values), f"grille {nom} : colonne {c} differente"
        for c in ("z_terrain_m", "z_rel_m") + (("z0_m", "slope_deg", "aspect_deg") if nom == "surface" else ()):
            e = float(np.nanmax(np.abs(a[c].values - b[c].values)))
            assert e < 1e-3, f"grille {nom} : colonne {c}, ecart max {e}"
    return meta2


def m1_vierge_run(ds):
    """Comme run() de la cellule 10i, avec en plus la masse et la rugosite AVANT correction (v7 seul)."""
    N = len(ds)
    R = {"avant": np.zeros((N, NZ, 2)), "apres": np.zeros((N, NZ, 2)),
         "st": np.zeros((N, NZ, 2)), "st2": np.zeros((N, NZ, 2)),
         "mass_avant": [], "mass_apres": [], "rough_avant": [], "rough_apres": []}
    for i, (s_raw, sensor, tfeat, y, _) in enumerate(DataLoader(ds, batch_size=1, num_workers=NUM_WORKERS)):
        u, v, w, uc, vc = predict_corrected(s_raw, sensor, tfeat)
        y = y.to(device)
        for k in EVAL_LEVELS:
            mk = VALID_T[k]
            for ci_ in range(2):
                tk = y[0, ci_, k][mk].double()
                R["avant"][i, k, ci_] = (((u, v)[ci_][0, k][mk].double() - tk) ** 2).sum().item()
                R["apres"][i, k, ci_] = (((uc, vc)[ci_][0, k][mk].double() - tk) ** 2).sum().item()
                R["st"][i, k, ci_], R["st2"][i, k, ci_] = tk.sum().item(), (tk ** 2).sum().item()
        for lab, (a, b) in (("avant", (u, v)), ("apres", (uc, vc))):
            r = mass_residual(y[:, 3], a, b, w)
            R["mass_" + lab].append(torch.sqrt((r[:, MASS_MASK_T] ** 2).mean()).item())
            R["rough_" + lab].append(((roughness(a) + roughness(b)) / (roughness(y[:, 0]) + roughness(y[:, 1]))).item())
    return R


def m1_vierge_mesure():
    global CACHE_DIR
    t0 = time.time()
    meta2 = m1_vierge_grille()
    sha_meta = m1_sha256(M1_VIERGE_DIR / "meta.json")
    sha_grille = m1_sha256(M1_VIERGE_DIR / "aneo_grid_points.csv")
    print(f"Geometrie identique au jeu v2 | sha256 meta.json {sha_meta} | grille {sha_grille} ({time.time() - t0:.0f} s)")
    cache_v2 = CACHE_DIR
    try:
        CACHE_DIR = M1_VIERGE_CACHE                    # lu par build_day (BLOC 3) et ANEO3D (BLOC 6)
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        idx_path = CACHE_DIR / "index.csv"
        faits = set(pd.read_csv(idx_path)["day"]) if idx_path.exists() else set()
        for path in sorted(M1_VIERGE_DIR.glob("aneo_observations_day_*_multitime.csv*")):
            if day_of(path) in faits:
                continue
            rows = pd.DataFrame(build_day(path)).assign(split="vierge")
            rows.to_csv(idx_path, mode="a", header=not idx_path.exists(), index=False)
            print(f"  cache jeu vierge : jour {day_of(path):03d}, {len(rows)} instants ({time.time() - t0:.0f} s)")
        idx = pd.read_csv(idx_path).drop_duplicates("stem").sort_values("stem").reset_index(drop=True)
        idx = idx[[is_synoptic(t) for t in idx["timestamp"]]].reset_index(drop=True)
        st_, hr_, dy_ = idx["stem"].values, idx["hour"].values, idx["doy"].values
        jours = idx["day"].values
        print(f"Jeu vierge : {len(idx)} instants synoptiques, {len(set(jours))} jours")
        R = m1_vierge_run(ANEO3D(st_, hr_, dy_, train=False, seed=2000))
        sub_v = np.unique(np.linspace(0, len(st_) - 1, min(40, len(st_))).astype(int))
        R25 = run_keep(sub_v, 3000, 0.25, stems=st_, hours=hr_, doys=dy_)
    finally:
        CACHE_DIR = cache_v2
    groupes = [np.nonzero(jours == d)[0] for d in pd.unique(jours)]
    rng = np.random.default_rng(2027)
    full, dr = boot(R, lambda: np.concatenate([groupes[g] for g in rng.integers(0, len(groupes), len(groupes))]))
    N = len(st_)
    n_all = N * NK[:, None]
    sst = R["st2"].sum(0) - R["st"].sum(0) ** 2 / n_all
    r2 = {n: 1 - R[n].sum(0) / sst for n in ("avant", "apres")}
    f25 = {n: rmse_lv(R25[n].sum(0), R25["avant"].shape[0]) for n in ("avant", "apres")}
    rows = []
    print("=" * 96)
    print(f"JEU VIERGE graine 2027 (HORS REGLE) : {N} instants, 50 % des capteurs, IC 95 % bootstrap par jours ({len(groupes)} grappes)")
    print(" Niveau |  u avant [IC]          u apres [IC]          R2 apres |  v avant [IC]          v apres [IC]          R2 apres")
    for k in LV:
        cells = []
        for ci_, c in enumerate("uv"):
            ia, ip = ci(dr["avant"][:, k, ci_]), ci(dr["apres"][:, k, ci_])
            dd = ci(dr["apres"][:, k, ci_] - dr["avant"][:, k, ci_])
            cells.append(f"{full['avant'][k, ci_]:.3f} [{ia[0]:.3f}, {ia[1]:.3f}]  {full['apres'][k, ci_]:.3f} "
                         f"[{ip[0]:.3f}, {ip[1]:.3f}]  {r2['apres'][k, ci_]:.4f}")
            rows.append({"z_m": int(Z_LEVELS[k]), "comp": c,
                         "avant": full["avant"][k, ci_], "avant_ic_bas": ia[0], "avant_ic_haut": ia[1],
                         "apres": full["apres"][k, ci_], "apres_ic_bas": ip[0], "apres_ic_haut": ip[1],
                         "delta": full["apres"][k, ci_] - full["avant"][k, ci_], "delta_ic_bas": dd[0], "delta_ic_haut": dd[1],
                         "r2_avant": r2["avant"][k, ci_], "r2_apres": r2["apres"][k, ci_],
                         "cap25_avant": f25["avant"][k, ci_], "cap25_apres": f25["apres"][k, ci_]})
        print(f" {int(Z_LEVELS[k]):5d} m |  {cells[0]} |  {cells[1]}")
    W = {}
    for ci_, c in enumerate("uv"):
        for n in ("avant", "apres"):
            W[f"{c}_{n}"], W[f"{c}_{n}_ic95"] = worst(full, dr, n, ci_)
        print(f"Pire niveau {c} : v7 seul {W[c + '_avant']:.3f} {tuple(round(x, 3) for x in W[c + '_avant_ic95'])}"
              f" -> v7 + correction {W[c + '_apres']:.3f} {tuple(round(x, 3) for x in W[c + '_apres_ic95'])} m/s")
    print("Difference appariee apres - avant, 650-1000 m (IC 95 %) :")
    for r_ in rows:
        if r_["z_m"] >= 650:
            print(f"  {r_['z_m']:5d} m {r_['comp']} : {r_['delta']:+.3f} [{r_['delta_ic_bas']:+.3f}, {r_['delta_ic_haut']:+.3f}]")
    dso = {n: {"dso2": float(np.mean(R["mass_" + n])), "dso3": float(np.mean(R["rough_" + n])),
               "dso5_25pct": float(f25[n][LV].max())} for n in ("avant", "apres")}
    for n, lab in (("avant", "v7 seul"), ("apres", "v7 + correction")):
        print(f"{lab:16s} : DSO2 {dso[n]['dso2']:.2e} | DSO3 {dso[n]['dso3']:.3f} | DSO5 (25 %, {len(sub_v)} instants) {dso[n]['dso5_25pct']:.3f}")
    print("Aucun seuil n'est applique ; aucune decision n'en depend (avenant n 2).")
    M1_VIERGE_OUT.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(M1_VIERGE_OUT / "par_niveau.csv", index=False)
    with open(M1_VIERGE_OUT / "summary.json", "w", encoding="utf-8") as f:
        json.dump({"hors_regle": True, "avenant": "n 2 (2026-10-14, salon #341)", "graine_jeu": 2027,
                   "sha256_meta_json": sha_meta, "sha256_grille": sha_grille, "calendrier": meta2["calendrier"],
                   "instants": int(N), "jours": sorted(int(d) for d in set(jours)), "capteurs_visibles": 0.5,
                   "graine_masques": 2000, "bootstrap": {"grappes": "jours", "tirages": B, "graine": 2027},
                   "pire_niveau": W, "dso": dso, "correction_sha256": CORR_SHA,
                   "instants_dso5": int(len(sub_v))}, f, indent=2)
    print("Resultats :", M1_VIERGE_OUT, f"({(time.time() - t0) / 60:.1f} min)")


_raison = m1_vierge_absent()
if _raison is not None:
    print("JEU VIERGE NON MESURE (hors regle, sans effet sur la decision 10i) : " + _raison)
    print(f"Pour le mesurer : deposer le jeu complet (meta.json, split_days.json, aneo_grid_points.csv, "
          f"{M1_VIERGE_JOURS} fichiers aneo_observations_day_*_multitime.csv.gz) dans {M1_VIERGE_DIR}, puis executer "
          "cette cellule seule dans la meme session (apres 10i).")
elif (M1_VIERGE_OUT / "summary.json").exists():
    print(f"Jeu vierge deja mesure ({M1_VIERGE_OUT / 'summary.json'}) : une seule execution (avenant n 2).")
else:
    m1_vierge_mesure()
print("M1-f OK")
